In [1]:
#!/usr/bin/env python3
"""
MAF Metrics Applied to DREAM Cloud Scheduling Strategies
=========================================================

Compares three pointing strategies on a single night (default 2025-07-15):
  Green  – Absolute minimum cloud extinction
  Blue   – Cloud motion tracking prediction
  Red    – Baseline scheduler (opsim .db file)

For each strategy, this script constructs a synthetic opsim-like visit table
from the DREAM cloud maps, then runs a suite of MAF metrics:

  Survey coverage / efficiency
    - CountMetric          : number of valid exposures
    - OpenShutterFractionMetric : fraction of time actually exposing
    - SumMetric            : total photon proxy (fiveSigmaDepth proxy)

  Airmass / observing conditions
    - MedianMetric(airmass)
    - MaxMetric(airmass)
    - MeanMetric(seeingFwhmEff) – estimated from extinction proxy

  Survey depth
    - Coaddm5Metric        : coadded 5-sigma depth map (HealpixSlicer)
    - MedianMetric(fiveSigmaDepth) – per-strategy scalar

  Cadence / gaps
    - MaxGapMetric         : largest gap between exposures
    - MeanMetric(slewTime) : mean slew time

  Sky coverage
    - fO metrics (fraction of sky above Nvis threshold)
    - UniSlicer summary statistics

All results are printed as a comparison table and saved as:
  - maf_scheduling_comparison.png   : summary metric bar charts
  - maf_sky_maps.png                : per-strategy Healpix depth maps
  - maf_results_summary.csv         : raw metric values

Usage
-----
  python maf_scheduling_comparison.py \
      --csv    feb5_data.csv \
      --db     baseline_v5.1.0_10yrs.db \
      --date   2025-07-15 \
      [--max-frames 200]

Requirements
------------
  rubin_sim (maf), rubin_scheduler, healpy, astropy, h5py, lsst.resources
"""

import argparse
import io
import os
import sqlite3
import warnings
from datetime import timedelta

import h5py
import healpy as hp
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.ndimage import gaussian_filter
from scipy import ndimage
from astropy.coordinates import SkyCoord, EarthLocation, AltAz
from astropy.time import Time
import astropy.units as u

warnings.filterwarnings("ignore")

# ── optional LSST imports ─────────────────────────────────────────────────────
try:
    from lsst.resources import ResourcePath
    HAS_LSST_RESOURCES = True
except ImportError:
    HAS_LSST_RESOURCES = False
    print("WARNING: lsst.resources not found – using fallback HTTP fetch.")

try:
    import rubin_sim.maf as maf
    HAS_MAF = True
    print("rubin_sim.maf available – running full MAF suite.")
except ImportError:
    HAS_MAF = False
    print("WARNING: rubin_sim.maf not found – running lightweight metric equivalents.")

# ═══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ═══════════════════════════════════════════════════════════════════════════════
DEFAULT_CSV        = "feb5_data.csv"
DEFAULT_DB         = "baseline_v5.1.0_10yrs.db"
DEFAULT_DATE       = "2025-07-15"
OUTPUT_PNG         = "maf_scheduling_comparison.png"
OUTPUT_MAPS_PNG    = "maf_sky_maps.png"
OUTPUT_CSV         = "maf_results_summary.csv"

NSIDE              = 32
NEST               = True
RUBIN_LAT          = -30.244639
RUBIN_LON          = -70.749417
RUBIN_HEIGHT_M     = 2663.0
R_PROJECTION       = 10_000.0        # km
BIN_SIZE_KM        = 1_000
MIN_ALT_DEG        = 15.0
MAX_ZA_DEG         = 70.0

# Photometry
RUBIN_AREA         = np.pi * (6.4 / 2) ** 2   # m²
RUBIN_QE           = 0.8
EXPOSURE_TIME      = 30.0                       # s
READOUT_TIME       = 2.0                        # s
SLOT_TIME          = EXPOSURE_TIME + READOUT_TIME
PHOTON_FLUX_MAG20  = 100.0                      # photons/s/m² at zenith

# Slew model (Rubin simplified)
MAX_SLEW_ALT       = 3.5    # deg/s
MAX_SLEW_AZ        = 7.0    # deg/s
SLEW_SETTLE        = 2.0    # s

# MAF sky-coverage threshold
MIN_NVIS_FO        = 825    # fO benchmark: N visits over footprint

# Strategy colours / labels
STRATEGIES   = ("absolute", "motion", "scheduler")
STRAT_COLORS = {"absolute": "#2ca02c", "motion": "#1f77b4", "scheduler": "#d62728"}
STRAT_LABELS = {"absolute": "Abs Min (Green)",
                "motion":   "Motion Track (Blue)",
                "scheduler":"Scheduler (Red)"}

URL_COL      = "lsst.sal.DREAM.logevent_largeFileObjectAvailable.url"
TIME_COL     = "time"


# ═══════════════════════════════════════════════════════════════════════════════
# UTILITY HELPERS
# ═══════════════════════════════════════════════════════════════════════════════

def _location():
    return EarthLocation(lat=RUBIN_LAT*u.deg, lon=RUBIN_LON*u.deg,
                         height=RUBIN_HEIGHT_M*u.m)


def _ensure_time(t):
    if not isinstance(t, Time):
        t = Time(t)
    return t.utc


def transform_url(url):
    url = str(url).strip()
    if url.startswith("https://s3.cp.lsst.org/"):
        return url.replace("https://s3.cp.lsst.org/", "s3://lfa@")
    return url


def fetch_cloud_map(url):
    """Load cloud_sys HDF5 → (clouds, sigma) arrays, bad pixels → NaN."""
    if HAS_LSST_RESOURCES:
        rp = ResourcePath(url)
        with rp.open("rb") as fd:
            data = fd.read()
    else:
        import urllib.request
        with urllib.request.urlopen(url) as resp:
            data = resp.read()

    with h5py.File(io.BytesIO(data), "r") as f:
        clouds = np.array(f["clouds"],       dtype=float).ravel()
        sigma  = np.array(f["sigma"],        dtype=float).ravel()
        flags  = np.array(f["flags"],        dtype=int  ).ravel()
        vis    = np.array(f["mask_visible"], dtype=bool ).ravel()
        nobs   = np.array(f["nobs"],         dtype=int  ).ravel()

    bad = ~vis | (nobs == 0) | (flags > 0) | (sigma > 0.3) | ~np.isfinite(clouds)
    clouds[bad] = np.nan
    sigma[bad]  = np.nan
    return clouds, sigma


def healpix_to_grid(clouds, obstime):
    """Project HEALPix map onto flat-sky km grid (30×30 bins of BIN_SIZE_KM)."""
    npix = hp.nside2npix(NSIDE)
    pix  = np.arange(npix)
    th, ph = hp.pix2ang(NSIDE, pix, nest=NEST)
    ra  = np.degrees(ph)
    dec = 90.0 - np.degrees(th)

    t   = _ensure_time(obstime)
    loc = _location()
    sky = SkyCoord(ra=ra*u.deg, dec=dec*u.deg, frame="icrs")
    aa  = sky.transform_to(AltAz(obstime=t, location=loc))
    alt = aa.alt.deg
    az  = aa.az.deg % 360.0

    alt_r  = np.radians(alt)
    az_r   = np.radians(az)
    above  = alt > MIN_ALT_DEG
    scale  = np.where(above, R_PROJECTION / np.sin(alt_r), np.nan)
    xf     = -np.cos(alt_r) * np.sin(az_r) * scale
    yf     =  np.cos(alt_r) * np.cos(az_r) * scale
    vals   = np.where(above, clouds, np.nan)

    ok = ~np.isnan(vals) & (np.sqrt(xf**2 + yf**2) <= 15_000)
    x_edges = np.arange(-15_000, 15_001, BIN_SIZE_KM)
    y_edges = np.arange(-15_000, 15_001, BIN_SIZE_KM)

    Hs, _, _ = np.histogram2d(xf[ok], yf[ok], bins=[x_edges, y_edges], weights=vals[ok])
    Hc, _, _ = np.histogram2d(xf[ok], yf[ok], bins=[x_edges, y_edges])
    with np.errstate(divide="ignore", invalid="ignore"):
        H = Hs / Hc
    H[Hc == 0] = np.nan
    H = H.T
    xc = (x_edges[:-1] + x_edges[1:]) / 2
    yc = (y_edges[:-1] + y_edges[1:]) / 2
    Xg, Yg = np.meshgrid(xc, yc)
    H[np.sqrt(Xg**2 + Yg**2) > 15_000] = np.nan
    return H, x_edges, y_edges


def xy_to_altaz(x_km, y_km):
    r   = np.sqrt(x_km**2 + y_km**2)
    alt = float(np.degrees(np.arctan2(R_PROJECTION, r)))
    az  = float(np.degrees(np.arctan2(-x_km, y_km)) % 360.0)
    return alt, az


def altaz_to_radec(alt_deg, az_deg, obstime):
    t   = _ensure_time(obstime)
    loc = _location()
    aa  = SkyCoord(alt=alt_deg*u.deg, az=az_deg*u.deg,
                   frame=AltAz(obstime=t, location=loc))
    icrs = aa.icrs
    return float(icrs.ra.deg), float(icrs.dec.deg)


def grid_value(grid, x_km, y_km):
    xi = int(round(x_km / BIN_SIZE_KM + 15))
    yi = int(round(y_km / BIN_SIZE_KM + 15))
    if 0 <= xi < grid.shape[1] and 0 <= yi < grid.shape[0]:
        return grid[yi, xi]
    return np.nan


def find_abs_min(grid):
    if not np.any(~np.isnan(grid)):
        return 0, 0, np.nan
    idx     = np.nanargmin(grid)
    yi, xi  = np.unravel_index(idx, grid.shape)
    return (xi - 15) * BIN_SIZE_KM, (yi - 15) * BIN_SIZE_KM, grid[yi, xi]


def radec_to_xy(ra_deg, dec_deg, obstime):
    t   = _ensure_time(obstime)
    loc = _location()
    sky = SkyCoord(ra=ra_deg*u.deg, dec=dec_deg*u.deg, frame="icrs")
    aa  = sky.transform_to(AltAz(obstime=t, location=loc))
    alt, az = float(aa.alt.deg), float(aa.az.deg % 360.0)
    if alt < MIN_ALT_DEG:
        return None, None, alt, az
    alt_r, az_r = np.radians(alt), np.radians(az)
    scale = R_PROJECTION / np.sin(alt_r)
    x = -np.cos(alt_r) * np.sin(az_r) * scale
    y =  np.cos(alt_r) * np.cos(az_r) * scale
    return float(x), float(y), alt, az


def slew_time(alt1, az1, alt2, az2):
    da  = abs(alt2 - alt1)
    daz = abs(az2 - az1)
    if daz > 180:
        daz = 360 - daz
    return max(da / MAX_SLEW_ALT, daz / MAX_SLEW_AZ) + SLEW_SETTLE


def extinction_to_m5(ext_mag, airmass=1.2, base_m5=24.5):
    """Rough 5-sigma depth estimate from cloud extinction and airmass."""
    return base_m5 - ext_mag - 0.1 * (airmass - 1.0)


def photon_count(ext_mag):
    if np.isnan(ext_mag):
        return np.nan
    flux = 10 ** (-0.4 * ext_mag)
    return PHOTON_FLUX_MAG20 * RUBIN_AREA * RUBIN_QE * flux * EXPOSURE_TIME


def cloud_motion(g1, g2, sigma=5.0, search=10):
    g1s = gaussian_filter(np.nan_to_num(g1, nan=0), sigma)
    g2s = gaussian_filter(np.nan_to_num(g2, nan=0), sigma)
    m1  = ~np.isnan(g1)
    m2  = ~np.isnan(g2)
    best, bdx, bdy = -np.inf, 0, 0
    for dy in range(-search, search + 1):
        for dx in range(-search, search + 1):
            sh = ndimage.shift(g2s, (dy, dx), order=1, mode="constant", cval=0)
            sm = ndimage.shift(m2.astype(float), (dy, dx), order=0, mode="constant", cval=0) > 0.5
            v  = m1 & sm
            if v.sum() < 100:
                continue
            v1, v2 = g1s[v], sh[v]
            if np.std(v1) > 0 and np.std(v2) > 0:
                c = np.corrcoef(v1, v2)[0, 1]
                if c > best:
                    best, bdx, bdy = c, dx, dy
    return bdx, bdy, best


# ═══════════════════════════════════════════════════════════════════════════════
# DATA LOADING
# ═══════════════════════════════════════════════════════════════════════════════

def load_night_frames(csv_file, target_date):
    """Return cloud_sys HDF5 rows for the given observing night."""
    print(f"\nLoading DREAM CSV: {csv_file}")
    df = pd.read_csv(csv_file)
    df.columns = df.columns.str.replace('"', "").str.strip()
    df = df.dropna(subset=[URL_COL]).copy()
    df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors="coerce", utc=True)
    df = df.dropna(subset=[TIME_COL])
    df = df[df[URL_COL].str.contains(r"\.hdf5",    case=False, na=False, regex=True)]
    df = df[df[URL_COL].str.contains("cloud_sys",  case=False, na=False)]
    df = df.sort_values(TIME_COL).reset_index(drop=True)

    tgt   = pd.to_datetime(target_date).date()
    dates = df[TIME_COL].dt.date
    next_ = tgt + timedelta(days=1)
    mask  = (dates == tgt) | ((dates == next_) & (df[TIME_COL].dt.hour < 12))
    night = df[mask].reset_index(drop=True)
    print(f"  {len(night)} cloud_sys frames for night {target_date}")
    if len(night):
        print(f"  {night.iloc[0][TIME_COL]} → {night.iloc[-1][TIME_COL]}")
    return night


def load_scheduler_db(db_file):
    """Load one full night of scheduler pointings from an opsim SQLite file."""
    print(f"\nLoading scheduler DB: {db_file}")
    conn   = sqlite3.connect(db_file)
    cursor = conn.cursor()
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = [r[0] for r in cursor.fetchall()]
    table  = next((n for n in ["observations", "SummaryAllProps", "Summary", "obs"]
                   if n in tables), tables[0])
    obs    = pd.read_sql_query(f"SELECT * FROM {table}", conn)
    conn.close()

    # Normalise column names
    night_col = next((c for c in obs.columns if "night" in c.lower()), None)
    obs["night"] = obs[night_col].astype(int)
    for want, candidates in {
        "observationStartMJD": ["observationStartMJD", "observationstartmjd", "mjd"],
        "fieldRA":  ["fieldRA",  "fieldra",  "ra"],
        "fieldDec": ["fieldDec", "fielddec", "dec"],
        "filter":   ["filter",   "band"],
        "airmass":  ["airmass"],
        "seeingFwhmEff": ["seeingFwhmEff", "seeingfwhmeff", "seeing"],
        "fiveSigmaDepth": ["fiveSigmaDepth", "fivesigmadepth", "m5"],
        "slewTime":       ["slewTime", "slewtime"],
        "visitExposureTime": ["visitExposureTime", "visitexposuretime", "exptime"],
    }.items():
        if want not in obs.columns:
            for c in candidates:
                if c in obs.columns:
                    obs[want] = obs[c]
                    break

    # Select best night: most observations over at least 4 h
    scores = {}
    for n in obs["night"].unique()[:500]:
        sub = obs[obs["night"] == n]
        if len(sub) < 200:
            continue
        dur = (sub["observationStartMJD"].max() - sub["observationStartMJD"].min()) * 24
        if dur < 4:
            continue
        scores[n] = len(sub) * 10 + dur

    chosen = max(scores, key=scores.get) if scores else obs.groupby("night").size().idxmax()
    night  = obs[obs["night"] == chosen].sort_values("observationStartMJD").reset_index(drop=True)
    dur    = (night["observationStartMJD"].max() - night["observationStartMJD"].min()) * 24
    print(f"  Scheduler night {chosen}: {len(night)} obs, {dur:.1f} h")
    return night


def match_sched_to_frames(sched, frame_times):
    """Map each DREAM frame → nearest scheduler pointing by fractional night position."""
    n        = len(frame_times)
    f_fracs  = np.linspace(0, 1, n)
    mjd      = sched["observationStartMJD"].values
    rng      = mjd.max() - mjd.min() + 1e-12
    s_fracs  = (mjd - mjd.min()) / rng
    return [sched.iloc[np.argmin(np.abs(s_fracs - f))] for f in f_fracs]


# ═══════════════════════════════════════════════════════════════════════════════
# BUILD VISIT TABLES (synthetic opsim rows) for each strategy
# ═══════════════════════════════════════════════════════════════════════════════

def build_visit_tables(df_night, sched_obs, max_frames=None):
    """
    Load every DREAM frame, run three strategies, and build three visit DataFrames
    that mimic an opsim output table for input to MAF or lightweight metrics.

    Columns produced (matching opsim schema):
      observationStartMJD, fieldRA, fieldDec, filter, airmass,
      fiveSigmaDepth, seeingFwhmEff, slewTime, visitExposureTime,
      clouds_ext, photons, night
    """
    if max_frames:
        df_night = df_night.iloc[:max_frames]

    matched = match_sched_to_frames(sched_obs, df_night[TIME_COL].tolist())

    rows    = {s: [] for s in STRATEGIES}
    prev_alt = {s: 90.0 for s in STRATEGIES}
    prev_az  = {s:  0.0 for s in STRATEGIES}

    # Cloud motion state
    prev_grid   = None
    xp, yp      = 0.0, 0.0        # motion-tracking position
    HL          = 3                # history length for motion averaging

    dx_hist, dy_hist = [], []

    print(f"\nBuilding visit tables for {len(df_night)} frames …")
    for idx, (_, row) in enumerate(df_night.iterrows()):
        if idx % 50 == 0:
            print(f"  Frame {idx}/{len(df_night)}")

        url     = transform_url(row[URL_COL])
        t_visit = row[TIME_COL]

        try:
            clouds, _ = fetch_cloud_map(url)
        except Exception as e:
            print(f"    skip frame {idx}: {e}")
            prev_grid = None
            continue

        grid, _, _ = healpix_to_grid(clouds, t_visit)

        # ── MJD (approximate) ──────────────────────────────────────────
        t_astropy = _ensure_time(t_visit)
        mjd       = float(t_astropy.mjd)

        # ── Absolute minimum ───────────────────────────────────────────
        xa, ya, ext_a = find_abs_min(grid)
        alt_a, az_a   = xy_to_altaz(xa, ya)
        if ext_a is not np.nan and not np.isnan(ext_a) and alt_a > MIN_ALT_DEG:
            slew_a = slew_time(prev_alt["absolute"], prev_az["absolute"], alt_a, az_a)
            ra_a, dec_a = altaz_to_radec(alt_a, az_a, t_visit)
            am_a = 1.0 / np.sin(np.radians(alt_a))
            m5_a = extinction_to_m5(ext_a, am_a)
            rows["absolute"].append(dict(
                observationStartMJD = mjd,
                fieldRA = ra_a, fieldDec = dec_a,
                filter  = "r", airmass = am_a,
                fiveSigmaDepth = m5_a,
                seeingFwhmEff  = 0.7 + 0.1 * (am_a - 1),
                slewTime       = slew_a,
                visitExposureTime = EXPOSURE_TIME,
                clouds_ext     = ext_a,
                photons        = photon_count(ext_a),
                night          = 0,
            ))
            prev_alt["absolute"], prev_az["absolute"] = alt_a, az_a

        # ── Motion tracking ────────────────────────────────────────────
        if prev_grid is not None:
            dx, dy, conf = cloud_motion(prev_grid, grid)
            if conf > 0.5:
                dx_hist.append(dx); dy_hist.append(dy)
            if len(dx_hist) > HL:
                dx_hist.pop(0); dy_hist.pop(0)
            dx_avg = float(np.mean(dx_hist)) if dx_hist else 0.0
            dy_avg = float(np.mean(dy_hist)) if dy_hist else 0.0
            # Move opposite to cloud motion
            xp_new = xp - dx_avg * BIN_SIZE_KM
            yp_new = yp - dy_avg * BIN_SIZE_KM
            r_new  = np.sqrt(xp_new**2 + yp_new**2)
            if r_new > 14_000:
                xp_new *= 14_000 / r_new
                yp_new *= 14_000 / r_new
            val_new = grid_value(grid, xp_new, yp_new)
            if np.isnan(val_new) or val_new > 0.5:
                xp_new, yp_new, _ = find_abs_min(grid)
            xp, yp = xp_new, yp_new
        else:
            xp, yp, _ = find_abs_min(grid)

        ext_p = grid_value(grid, xp, yp)
        alt_p, az_p = xy_to_altaz(xp, yp)
        if not np.isnan(ext_p) and alt_p > MIN_ALT_DEG:
            slew_p = slew_time(prev_alt["motion"], prev_az["motion"], alt_p, az_p)
            ra_p, dec_p = altaz_to_radec(alt_p, az_p, t_visit)
            am_p = 1.0 / np.sin(np.radians(alt_p))
            m5_p = extinction_to_m5(ext_p, am_p)
            rows["motion"].append(dict(
                observationStartMJD = mjd,
                fieldRA = ra_p, fieldDec = dec_p,
                filter  = "r", airmass = am_p,
                fiveSigmaDepth = m5_p,
                seeingFwhmEff  = 0.7 + 0.1 * (am_p - 1),
                slewTime       = slew_p,
                visitExposureTime = EXPOSURE_TIME,
                clouds_ext     = ext_p,
                photons        = photon_count(ext_p),
                night          = 0,
            ))
            prev_alt["motion"], prev_az["motion"] = alt_p, az_p

        prev_grid = grid.copy()

        # ── Scheduler ──────────────────────────────────────────────────
        obs     = matched[idx]
        xs, ys, alt_s, az_s = radec_to_xy(
            float(obs["fieldRA"]), float(obs["fieldDec"]), t_visit)
        if xs is not None:
            ext_s = grid_value(grid, xs, ys)
            if not np.isnan(ext_s) and alt_s > MIN_ALT_DEG:
                slew_s = slew_time(prev_alt["scheduler"], prev_az["scheduler"], alt_s, az_s)
                am_s   = 1.0 / np.sin(np.radians(alt_s))
                m5_s   = extinction_to_m5(ext_s, am_s)
                # Inherit realistic seeing/depth from opsim if available
                raw_m5 = obs.get("fiveSigmaDepth", m5_s)
                if pd.isna(raw_m5):
                    raw_m5 = m5_s
                rows["scheduler"].append(dict(
                    observationStartMJD = mjd,
                    fieldRA  = float(obs["fieldRA"]),
                    fieldDec = float(obs["fieldDec"]),
                    filter   = str(obs.get("filter", "r")).strip("'"),
                    airmass  = am_s,
                    fiveSigmaDepth  = float(raw_m5) - ext_s,   # apply cloud penalty
                    seeingFwhmEff   = float(obs.get("seeingFwhmEff", 0.8) or 0.8),
                    slewTime        = slew_s,
                    visitExposureTime = EXPOSURE_TIME,
                    clouds_ext      = ext_s,
                    photons         = photon_count(ext_s),
                    night           = 0,
                ))
                prev_alt["scheduler"], prev_az["scheduler"] = alt_s, az_s

    dfs = {s: pd.DataFrame(rows[s]) for s in STRATEGIES}
    for s, df in dfs.items():
        print(f"  {s:10s}: {len(df):4d} visits")
    return dfs


# ═══════════════════════════════════════════════════════════════════════════════
# LIGHTWEIGHT METRICS (no rubin_sim.maf required)
# ═══════════════════════════════════════════════════════════════════════════════

def lightweight_metrics(dfs):
    """
    Compute a comprehensive set of per-strategy scalar metrics without MAF.
    Returns a dict of {metric_name: {strategy: value}}.
    """
    results = {}

    for s, df in dfs.items():
        if df.empty:
            continue
        mjd  = df["observationStartMJD"].values
        sort_idx = np.argsort(mjd)
        mjd  = mjd[sort_idx]
        ph   = df["photons"].values[sort_idx]
        m5   = df["fiveSigmaDepth"].values[sort_idx]
        am   = df["airmass"].values[sort_idx]
        see  = df["seeingFwhmEff"].values[sort_idx]
        slew = df["slewTime"].values[sort_idx]
        ext  = df["clouds_ext"].values[sort_idx]

        # ── Count / efficiency ───────────────────────────────────────
        results.setdefault("N_visits",         {})[s] = len(df)
        results.setdefault("Total_photons",    {})[s] = float(np.nansum(ph))

        dur_h = (mjd[-1] - mjd[0]) * 24 if len(mjd) > 1 else 0.0
        total_slew_h = float(np.sum(slew)) / 3600.0
        results.setdefault("Night_duration_h", {})[s] = dur_h
        results.setdefault("Total_slew_h",     {})[s] = total_slew_h
        exp_h = len(df) * EXPOSURE_TIME / 3600.0
        results.setdefault("Total_exposure_h", {})[s] = exp_h
        open_frac = exp_h / (dur_h + 1e-12)
        results.setdefault("Open_shutter_frac",{})[s] = float(open_frac)

        # ── Airmass ──────────────────────────────────────────────────
        results.setdefault("Median_airmass",   {})[s] = float(np.nanmedian(am))
        results.setdefault("Max_airmass",      {})[s] = float(np.nanmax(am))
        results.setdefault("Mean_airmass",     {})[s] = float(np.nanmean(am))

        # ── Seeing ───────────────────────────────────────────────────
        results.setdefault("Median_seeing",    {})[s] = float(np.nanmedian(see))

        # ── Depth ────────────────────────────────────────────────────
        results.setdefault("Median_m5",        {})[s] = float(np.nanmedian(m5))
        results.setdefault("Mean_m5",          {})[s] = float(np.nanmean(m5))
        results.setdefault("Coadd_m5_proxy",   {})[s] = float(
            -2.5 * np.log10(np.nansum(10 ** (-0.4 * m5 * 2))) / 2
            if len(m5) else np.nan)

        # ── Cloud extinction ─────────────────────────────────────────
        results.setdefault("Mean_cloud_ext",   {})[s] = float(np.nanmean(ext))
        results.setdefault("Median_cloud_ext", {})[s] = float(np.nanmedian(ext))
        results.setdefault("Min_cloud_ext",    {})[s] = float(np.nanmin(ext))

        # ── Cadence / gaps ───────────────────────────────────────────
        if len(mjd) > 1:
            gaps = np.diff(mjd) * 24 * 60   # minutes
            results.setdefault("Max_gap_min",   {})[s] = float(gaps.max())
            results.setdefault("Median_gap_min",{})[s] = float(np.median(gaps))
        else:
            results.setdefault("Max_gap_min",   {})[s] = np.nan
            results.setdefault("Median_gap_min",{})[s] = np.nan

        # ── Slew ─────────────────────────────────────────────────────
        results.setdefault("Mean_slew_s",      {})[s] = float(np.nanmean(slew))
        results.setdefault("Median_slew_s",    {})[s] = float(np.nanmedian(slew))
        results.setdefault("Max_slew_s",       {})[s] = float(np.nanmax(slew))

        # ── Sky coverage (healpix) ────────────────────────────────────
        pix_set = set()
        for _, r in df.iterrows():
            th  = np.radians(90.0 - r["fieldDec"])
            phi = np.radians(r["fieldRA"] % 360.0)
            pix_set.add(hp.ang2pix(NSIDE, th, phi, nest=NEST))
        sky_frac = len(pix_set) / hp.nside2npix(NSIDE)
        results.setdefault("Sky_frac_visited", {})[s] = float(sky_frac)

        # ── Uniformity (Kuiper-style spatial spread) ─────────────────
        ra_vals  = np.radians(df["fieldRA"].values)
        dec_vals = df["fieldDec"].values
        cos_dec  = np.cos(np.radians(dec_vals))
        # Normalised spatial entropy over visited pixels
        counts   = np.zeros(hp.nside2npix(NSIDE))
        for p in pix_set:
            counts[p] = 1
        frac_vis = len(pix_set) / hp.nside2npix(NSIDE)
        results.setdefault("Uniformity_frac", {})[s] = float(frac_vis)

        # ── SRD-style fO proxy ────────────────────────────────────────
        # fraction of healpix pixels with ≥1 visit
        results.setdefault("fO_area_proxy",    {})[s] = float(len(pix_set) * hp.nside2pixarea(NSIDE, degrees=True))

        # ── Survey depth relative to photometric ─────────────────────
        phot_m5 = extinction_to_m5(0.0)    # extinction=0, baseline
        delta_depth = np.nanmean(m5) - phot_m5
        results.setdefault("Delta_depth_vs_phot", {})[s] = float(delta_depth)

    return results


# ═══════════════════════════════════════════════════════════════════════════════
# FULL MAF METRICS (when rubin_sim.maf available)
# ═══════════════════════════════════════════════════════════════════════════════

def run_maf_metrics(dfs, out_dir="maf_outputs"):
    """
    Run a full MAF metric suite over an in-memory opsim-like visit table.
    Uses rubin_sim.maf with a temporary SQLite database written from each
    strategy's visit DataFrame.
    """
    import tempfile, os
    os.makedirs(out_dir, exist_ok=True)

    # Columns required by MAF
    REQUIRED = [
        "observationId", "fieldRA", "fieldDec", "filter",
        "observationStartMJD", "visitExposureTime",
        "airmass", "seeingFwhmEff", "fiveSigmaDepth",
        "slewTime", "night",
        # MAF internals (can be zero)
        "rotSkyPos", "numExposures", "skyBrightness",
        "slewDistance", "altitude", "azimuth",
        "moonAlt", "sunAlt", "cloud",
    ]

    maf_results = {}

    for s, df in dfs.items():
        if df.empty:
            maf_results[s] = {}
            continue

        print(f"\n  Running MAF on {s} ({len(df)} visits) …")

        # ── Prepare full visit table ───────────────────────────────────
        full = df.copy()
        full["observationId"]   = np.arange(len(full), dtype=int)
        full["rotSkyPos"]       = 0.0
        full["numExposures"]    = 1
        full["skyBrightness"]   = 21.0
        full["slewDistance"]    = full["slewTime"] * 3.5  # rough deg
        full["altitude"]        = np.degrees(np.arcsin(1.0 / full["airmass"].clip(1, 5)))
        full["azimuth"]         = 180.0
        full["moonAlt"]         = -30.0
        full["sunAlt"]          = -18.0
        full["cloud"]           = full["clouds_ext"].fillna(0.5)
        for col in REQUIRED:
            if col not in full.columns:
                full[col] = 0.0

        # ── Write temp SQLite ──────────────────────────────────────────
        with tempfile.NamedTemporaryFile(suffix=".db", delete=False) as tmp:
            db_path = tmp.name

        conn = sqlite3.connect(db_path)
        full[REQUIRED].to_sql("observations", conn, if_exists="replace", index=False)
        conn.close()

        strat_out = os.path.join(out_dir, s)
        os.makedirs(strat_out, exist_ok=True)

        # ── Define metric bundles ──────────────────────────────────────
        uni   = maf.UniSlicer()
        healpix = maf.HealpixSlicer(nside=NSIDE)

        bundles = [
            # Count
            maf.MetricBundle(maf.CountMetric("observationId"),      uni, "", run_name=s),
            # Airmass distribution
            maf.MetricBundle(maf.MedianMetric("airmass"),            uni, "", run_name=s),
            maf.MetricBundle(maf.MaxMetric("airmass"),               uni, "", run_name=s),
            maf.MetricBundle(maf.MeanMetric("airmass"),              uni, "", run_name=s),
            # Seeing
            maf.MetricBundle(maf.MedianMetric("seeingFwhmEff"),      uni, "", run_name=s),
            # 5-sigma depth
            maf.MetricBundle(maf.MedianMetric("fiveSigmaDepth"),     uni, "", run_name=s),
            maf.MetricBundle(maf.MeanMetric("fiveSigmaDepth"),       uni, "", run_name=s),
            maf.MetricBundle(maf.MinMetric("fiveSigmaDepth"),        uni, "", run_name=s),
            # Slew
            maf.MetricBundle(maf.MeanMetric("slewTime"),             uni, "", run_name=s),
            maf.MetricBundle(maf.SumMetric("slewTime"),              uni, "", run_name=s),
            # Open-shutter fraction
            maf.MetricBundle(maf.OpenShutterFractionMetric(),        uni, "", run_name=s),
            # Gaps
            maf.MetricBundle(maf.MaxGapMetric(),                     uni, "", run_name=s),
            # Sky-map depth
            maf.MetricBundle(maf.Coaddm5Metric(),                healpix, "", run_name=s),
            # fO area
            maf.MetricBundle(
                maf.fOArea(col="observationId", nside=NSIDE),   healpix, "", run_name=s),
        ]

        bg = maf.MetricBundleGroup(bundles, db_path, out_dir=strat_out)
        bg.run_all()
        os.unlink(db_path)

        # Harvest scalar results
        scalars = {}
        for b in bundles:
            mv = b.metric_values
            if hasattr(mv, "compressed") and mv.compressed().size == 1:
                scalars[b.metric.name] = float(mv.compressed()[0])
            elif hasattr(mv, "data") and mv.data.size == 1:
                scalars[b.metric.name] = float(mv.data.ravel()[0])
        maf_results[s] = scalars

    return maf_results


# ═══════════════════════════════════════════════════════════════════════════════
# PRINT COMPARISON TABLE
# ═══════════════════════════════════════════════════════════════════════════════

def print_table(results, title="METRIC COMPARISON"):
    print("\n" + "═" * 72)
    print(f"  {title}")
    print("═" * 72)
    col_w = 20
    strat_w = 18

    hdr = f"{'Metric':<{col_w}}" + "".join(f"{STRAT_LABELS[s][:strat_w]:<{strat_w}}"
                                              for s in STRATEGIES)
    print(hdr)
    print("─" * len(hdr))

    for metric, vals in sorted(results.items()):
        row = f"{metric:<{col_w}}"
        best_val = None
        best_fn  = min   # lower is better by default
        if any(k in metric for k in ("photon", "N_vis", "m5", "exposure",
                                      "frac", "sky", "fO", "depth", "open")):
            best_fn = max
        try:
            numeric_vals = {k: v for k, v in vals.items() if not np.isnan(v)}
            if numeric_vals:
                best_val = best_fn(numeric_vals.values())
        except Exception:
            pass

        for s in STRATEGIES:
            v = vals.get(s, np.nan)
            fmt = f"{v:.4f}" if not np.isnan(v) else "  N/A  "
            marker = " ★" if (best_val is not None and
                              not np.isnan(v) and
                              abs(v - best_val) < 1e-9) else "  "
            row += f"{fmt}{marker}".ljust(strat_w)
        print(row)
    print("═" * 72)
    print("  ★ = best value for that metric")


# ═══════════════════════════════════════════════════════════════════════════════
# PLOTS
# ═══════════════════════════════════════════════════════════════════════════════

def plot_summary(results, dfs, target_date, output=OUTPUT_PNG):
    """12-panel summary figure comparing all three strategies."""
    print(f"\nGenerating summary plot → {output}")

    fig = plt.figure(figsize=(22, 18))
    fig.suptitle(
        f"MAF-Style Metric Comparison — Night {target_date}\n"
        "DREAM Cloud Scheduling Strategies",
        fontsize=15, weight="bold", y=0.98)

    gs = gridspec.GridSpec(4, 3, figure=fig, hspace=0.45, wspace=0.35)

    colors = [STRAT_COLORS[s] for s in STRATEGIES]
    labels = [STRAT_LABELS[s]  for s in STRATEGIES]

    def bar_panel(ax, metric_key, ylabel, title, higher_better=True, scale=1.0):
        vals   = [results.get(metric_key, {}).get(s, np.nan) * scale for s in STRATEGIES]
        finite = [v for v in vals if not np.isnan(v)]
        if not finite:
            ax.set_title(title + " (no data)"); return
        bars = ax.bar(labels, vals, color=colors, alpha=0.78,
                      edgecolor="black", linewidth=1.2)
        best = max(finite) if higher_better else min(finite)
        for bar, val in zip(bars, vals):
            if np.isnan(val):
                continue
            ax.text(bar.get_x() + bar.get_width()/2, val,
                    f"{val:.3f}", ha="center", va="bottom", fontsize=8.5, weight="bold")
            if abs(val - best) < 1e-9:
                bar.set_edgecolor("gold")
                bar.set_linewidth(3)
        ax.set_ylabel(ylabel, fontsize=9)
        ax.set_title(title, fontsize=10, weight="bold")
        ax.grid(axis="y", alpha=0.3)
        ax.tick_params(axis="x", labelsize=7.5, rotation=15)

    # Row 0
    bar_panel(fig.add_subplot(gs[0, 0]), "N_visits",         "Count",     "N Visits",          True)
    bar_panel(fig.add_subplot(gs[0, 1]), "Total_photons",    "Photons",   "Total Photons",     True, 1e-6)
    bar_panel(fig.add_subplot(gs[0, 2]), "Open_shutter_frac","Fraction",  "Open-Shutter Frac", True)

    # Row 1
    bar_panel(fig.add_subplot(gs[1, 0]), "Median_m5",        "mag",       "Median 5σ Depth",   True)
    bar_panel(fig.add_subplot(gs[1, 1]), "Median_airmass",   "Airmass",   "Median Airmass",    False)
    bar_panel(fig.add_subplot(gs[1, 2]), "Mean_cloud_ext",   "mag",       "Mean Cloud Extinction", False)

    # Row 2
    bar_panel(fig.add_subplot(gs[2, 0]), "Mean_slew_s",      "s",         "Mean Slew Time",    False)
    bar_panel(fig.add_subplot(gs[2, 1]), "Total_slew_h",     "hours",     "Total Slew Time",   False)
    bar_panel(fig.add_subplot(gs[2, 2]), "fO_area_proxy",    "deg²",      "Sky Area Visited",  True)

    # Row 3: time-series photons & m5 histograms
    ax_ts  = fig.add_subplot(gs[3, :2])
    ax_m5h = fig.add_subplot(gs[3, 2])

    for s in STRATEGIES:
        df = dfs[s]
        if df.empty:
            continue
        mjd = df["observationStartMJD"].values
        ph  = df["photons"].values
        t_h = (mjd - mjd.min()) * 24
        ax_ts.plot(t_h, ph, color=STRAT_COLORS[s], alpha=0.6, lw=1.2,
                   label=STRAT_LABELS[s])

        m5  = df["fiveSigmaDepth"].dropna()
        ax_m5h.hist(m5, bins=25, color=STRAT_COLORS[s], alpha=0.5,
                    label=STRAT_LABELS[s], histtype="stepfilled", edgecolor="black", lw=0.8)

    ax_ts.set_xlabel("Hours into night", fontsize=9)
    ax_ts.set_ylabel("Photons / 30 s exposure", fontsize=9)
    ax_ts.set_title("Photon Count Per Exposure Over Night", fontsize=10, weight="bold")
    ax_ts.legend(fontsize=7); ax_ts.grid(alpha=0.3)
    ax_ts.ticklabel_format(axis="y", style="scientific", scilimits=(0,0))

    ax_m5h.set_xlabel("5σ Limiting Magnitude", fontsize=9)
    ax_m5h.set_ylabel("Count", fontsize=9)
    ax_m5h.set_title("Survey Depth Distribution", fontsize=10, weight="bold")
    ax_m5h.legend(fontsize=7); ax_m5h.grid(alpha=0.3)

    plt.savefig(output, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  → {output}")


def plot_sky_maps(dfs, target_date, output=OUTPUT_MAPS_PNG):
    """Healpix sky maps of mean 5-sigma depth for each strategy."""
    print(f"\nGenerating sky maps → {output}")
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    fig.suptitle(f"5σ Depth Sky Maps — Night {target_date}", fontsize=13, weight="bold")

    for ax, s in zip(axes, STRATEGIES):
        df  = dfs[s]
        sky = np.full(hp.nside2npix(NSIDE), np.nan)

        if not df.empty:
            for _, r in df.iterrows():
                th  = np.radians(90.0 - r["fieldDec"])
                phi = np.radians(r["fieldRA"] % 360.0)
                p   = hp.ang2pix(NSIDE, th, phi, nest=NEST)
                if np.isnan(sky[p]):
                    sky[p] = r["fiveSigmaDepth"]
                else:
                    # Stack: add in flux, then reconvert
                    sky[p] = -2.5 * np.log10(
                        10**(-0.4 * sky[p]) + 10**(-0.4 * r["fiveSigmaDepth"]))

        sky_ring = hp.reorder(sky, n2r=True) if NEST else sky
        sky_ring[np.isnan(sky_ring)] = hp.UNSEEN

        hp.mollview(sky_ring, fig=fig, sub=(1, 3, list(STRATEGIES).index(s) + 1),
                    title=STRAT_LABELS[s], unit="5σ mag",
                    min=22, max=26, cmap="viridis", hold=True)
        ax.remove()

    plt.savefig(output, dpi=130, bbox_inches="tight")
    plt.close()
    print(f"  → {output}")


def plot_maf_results(maf_results, output="maf_native_metrics.png"):
    """Bar chart of native MAF scalar metrics."""
    all_metrics = sorted(set(k for v in maf_results.values() for k in v))
    n = len(all_metrics)
    if n == 0:
        return

    ncols = 3
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(18, 4 * nrows))
    fig.suptitle("Native MAF Metrics by Strategy", fontsize=14, weight="bold")
    axes = axes.ravel()

    colors = [STRAT_COLORS[s] for s in STRATEGIES]
    labels = [STRAT_LABELS[s]  for s in STRATEGIES]

    for i, metric in enumerate(all_metrics):
        ax   = axes[i]
        vals = [maf_results.get(s, {}).get(metric, np.nan) for s in STRATEGIES]
        bars = ax.bar(labels, vals, color=colors, alpha=0.75,
                      edgecolor="black", linewidth=1.1)
        for bar, v in zip(bars, vals):
            if not np.isnan(v):
                ax.text(bar.get_x() + bar.get_width()/2, v,
                        f"{v:.3g}", ha="center", va="bottom", fontsize=8)
        ax.set_title(metric[:40], fontsize=9, weight="bold")
        ax.grid(axis="y", alpha=0.3)
        ax.tick_params(axis="x", labelsize=7, rotation=20)

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    plt.tight_layout()
    plt.savefig(output, dpi=130, bbox_inches="tight")
    plt.close()
    print(f"  MAF native plot → {output}")


# ═══════════════════════════════════════════════════════════════════════════════
# MAIN
# ═══════════════════════════════════════════════════════════════════════════════

def main(
    csv=DEFAULT_CSV,
    db=DEFAULT_DB,
    date=DEFAULT_DATE,
    max_frames=None,
    out_dir=".",
    _args=None,           # internal: pre-parsed namespace (overrides all)
):
    """
    Entry point — callable directly from Python/Jupyter or from the command line.

    Jupyter / IPython usage
    -----------------------
        from maf_scheduling_comparison import main
        main(csv="feb5_data.csv", db="baseline_v5.1.0_10yrs.db",
             date="2025-07-15", max_frames=200)

    Command-line usage
    ------------------
        python maf_scheduling_comparison.py \\
            --csv feb5_data.csv --db baseline_v5.1.0_10yrs.db \\
            --date 2025-07-15 --max-frames 200
    """
    # ── Argument resolution ───────────────────────────────────────────────────
    # When called from the CLI, _args is populated by _cli_main().
    # When called directly (Jupyter / import), keyword args are used as-is.
    if _args is not None:
        csv        = _args.csv
        db         = _args.db
        date       = _args.date
        max_frames = _args.max_frames
        out_dir    = _args.out_dir

    # Convenience: expose as a simple namespace so the rest of the function
    # uses attribute access uniformly.
    class _A:
        pass
    args = _A()
    args.csv        = csv
    args.db         = db
    args.date       = date
    args.max_frames = max_frames
    args.out_dir    = out_dir

    os.makedirs(args.out_dir, exist_ok=True)
    png_main  = os.path.join(args.out_dir, OUTPUT_PNG)
    png_maps  = os.path.join(args.out_dir, OUTPUT_MAPS_PNG)
    csv_out   = os.path.join(args.out_dir, OUTPUT_CSV)
    png_maf   = os.path.join(args.out_dir, "maf_native_metrics.png")

    print("\n" + "═" * 72)
    print("  MAF METRICS ON DREAM CLOUD SCHEDULING STRATEGIES")
    print(f"  Night: {args.date}")
    print("═" * 72)

    # 1. Load data
    df_night  = load_night_frames(args.csv, args.date)
    if df_night.empty:
        print(f"ERROR: No DREAM frames found for {args.date}.  Exiting.")
        return

    sched_obs = load_scheduler_db(args.db)

    # 2. Build synthetic visit tables
    dfs = build_visit_tables(df_night, sched_obs, max_frames=args.max_frames)

    # 3. Run lightweight metrics (always available)
    print("\n" + "═" * 50)
    print("  Running lightweight metrics …")
    lw_results = lightweight_metrics(dfs)
    print_table(lw_results, "LIGHTWEIGHT METRIC COMPARISON")

    # 4. Run full MAF metrics if available
    maf_results = {}
    if HAS_MAF:
        print("\n" + "═" * 50)
        print("  Running native MAF metrics …")
        try:
            maf_results = run_maf_metrics(dfs, out_dir=os.path.join(args.out_dir, "maf_outputs"))
            print_table(maf_results, "NATIVE MAF METRIC COMPARISON")
            plot_maf_results(maf_results, output=png_maf)
        except Exception as e:
            print(f"  MAF run failed: {e}")
            import traceback; traceback.print_exc()
    else:
        print("\n  (skipping native MAF – rubin_sim.maf not installed)")

    # 5. Combine results
    combined = {**lw_results, **maf_results}

    # 6. Save CSV
    rows = []
    for metric, vals in sorted(combined.items()):
        row = {"metric": metric}
        row.update(vals)
        rows.append(row)
    pd.DataFrame(rows).to_csv(csv_out, index=False)
    print(f"\n  Results saved → {csv_out}")

    # 7. Plots
    plot_summary(lw_results, dfs, args.date, output=png_main)
    try:
        plot_sky_maps(dfs, args.date, output=png_maps)
    except Exception as e:
        print(f"  Sky map failed (healpy mollview issue): {e}")

    # 8. Final scorecard
    print("\n" + "═" * 72)
    print("  FINAL SCORECARD  (best strategy per metric)")
    print("═" * 72)
    wins = {s: 0 for s in STRATEGIES}
    for metric, vals in lw_results.items():
        finite = {k: v for k, v in vals.items() if not np.isnan(v)}
        if not finite:
            continue
        higher_better = any(k in metric for k in (
            "photon", "N_vis", "m5", "exposure", "frac", "sky", "fO",
            "depth", "open", "Uniformity"))
        best_fn = max if higher_better else min
        winner  = best_fn(finite, key=finite.get)
        wins[winner] += 1
        print(f"  {metric:<28s}  → {STRAT_LABELS[winner]}")

    print("\n  Wins per strategy:")
    for s in STRATEGIES:
        print(f"    {STRAT_LABELS[s]:<30s}: {wins[s]:2d}")
    print("═" * 72)
    print("\nDone.")



def _cli_main():
    """CLI entry-point — strips Jupyter kernel args before parsing."""
    import sys
    parser = argparse.ArgumentParser(
        description="MAF metrics on DREAM cloud scheduling strategies")
    parser.add_argument("--csv",        default=DEFAULT_CSV)
    parser.add_argument("--db",         default=DEFAULT_DB)
    parser.add_argument("--date",       default=DEFAULT_DATE)
    parser.add_argument("--max-frames", type=int, default=None, dest="max_frames")
    parser.add_argument("--out-dir",    default=".",            dest="out_dir")

    # Strip unknown flags injected by Jupyter (e.g. -f <kernel-json>)
    known_flags = {"--csv", "--db", "--date", "--max-frames", "--out-dir", "-h", "--help"}
    clean, skip = [], False
    for tok in sys.argv[1:]:
        if skip:
            skip = False
            continue
        if tok in known_flags or any(tok.startswith(f + "=") for f in known_flags):
            clean.append(tok)
        elif tok.startswith("-") and tok not in known_flags:
            skip = True   # drop this flag AND its value
        else:
            clean.append(tok)

    main(_args=parser.parse_args(clean))


if __name__ == "__main__":
    _cli_main()

rubin_sim.maf available – running full MAF suite.

════════════════════════════════════════════════════════════════════════
  MAF METRICS ON DREAM CLOUD SCHEDULING STRATEGIES
  Night: 2025-07-15
════════════════════════════════════════════════════════════════════════

Loading DREAM CSV: feb5_data.csv
  2529 cloud_sys frames for night 2025-07-15
  2025-07-15 00:00:12.958000+00:00 → 2025-07-16 10:52:31.292000+00:00

Loading scheduler DB: baseline_v5.1.0_10yrs.db
  Scheduler night 572: 1139 obs, 11.7 h

Building visit tables for 2529 frames …
  Frame 0/2529
  Frame 50/2529
  Frame 100/2529
  Frame 150/2529
  Frame 200/2529
  Frame 250/2529
  Frame 300/2529
  Frame 350/2529
  Frame 400/2529
  Frame 450/2529
  Frame 500/2529
  Frame 550/2529
  Frame 600/2529
  Frame 650/2529
  Frame 700/2529
  Frame 750/2529
  Frame 800/2529
  Frame 850/2529
  Frame 900/2529
  Frame 950/2529
  Frame 1000/2529
  Frame 1050/2529
  Frame 1100/2529
  Frame 1150/2529
  Frame 1200/2529
  Frame 1250/2529
  Frame 1

Traceback (most recent call last):
  File "/tmp/ipykernel_4310/260488261.py", line 1054, in main
    maf_results = run_maf_metrics(dfs, out_dir=os.path.join(args.out_dir, "maf_outputs"))
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_4310/260488261.py", line 748, in run_maf_metrics
    maf.fOArea(col="observationId", nside=NSIDE),   healpix, "", run_name=s),
    ^^^^^^^^^^
AttributeError: module 'rubin_sim.maf' has no attribute 'fOArea'. Did you mean: 'FOArea'?


  → ./maf_scheduling_comparison.png

Generating sky maps → ./maf_sky_maps.png
  Sky map failed (healpy mollview issue): <Axes: >

════════════════════════════════════════════════════════════════════════
  FINAL SCORECARD  (best strategy per metric)
════════════════════════════════════════════════════════════════════════
  N_visits                      → Abs Min (Green)
  Total_photons                 → Abs Min (Green)
  Night_duration_h              → Scheduler (Red)
  Total_slew_h                  → Scheduler (Red)
  Total_exposure_h              → Abs Min (Green)
  Open_shutter_frac             → Abs Min (Green)
  Median_airmass                → Scheduler (Red)
  Max_airmass                   → Scheduler (Red)
  Mean_airmass                  → Scheduler (Red)
  Median_seeing                 → Motion Track (Blue)
  Median_m5                     → Abs Min (Green)
  Mean_m5                       → Abs Min (Green)
  Coadd_m5_proxy                → Abs Min (Green)
  Mean_cloud_ext        

In [2]:
#!/usr/bin/env python3
"""
MAF Metrics Applied to DREAM Cloud Scheduling Strategies
=========================================================

Compares three pointing strategies on a single night (default 2025-07-15):
  Green  – Absolute minimum cloud extinction
  Blue   – Cloud motion tracking prediction
  Red    – Baseline scheduler (opsim .db file)

For each strategy, this script constructs a synthetic opsim-like visit table
from the DREAM cloud maps, then runs a suite of MAF metrics:

  Survey coverage / efficiency
    - CountMetric          : number of valid exposures
    - OpenShutterFractionMetric : fraction of time actually exposing
    - SumMetric            : total photon proxy (fiveSigmaDepth proxy)

  Airmass / observing conditions
    - MedianMetric(airmass)
    - MaxMetric(airmass)
    - MeanMetric(seeingFwhmEff) – estimated from extinction proxy

  Survey depth
    - Coaddm5Metric        : coadded 5-sigma depth map (HealpixSlicer)
    - MedianMetric(fiveSigmaDepth) – per-strategy scalar

  Cadence / gaps
    - MaxGapMetric         : largest gap between exposures
    - MeanMetric(slewTime) : mean slew time

  Sky coverage
    - fO metrics (fraction of sky above Nvis threshold)
    - UniSlicer summary statistics

All results are printed as a comparison table and saved as:
  - maf_scheduling_comparison.png   : summary metric bar charts
  - maf_sky_maps.png                : per-strategy Healpix depth maps
  - maf_results_summary.csv         : raw metric values

Usage
-----
  python maf_scheduling_comparison.py \
      --csv    feb5_data.csv \
      --db     baseline_v5.1.0_10yrs.db \
      --date   2025-07-15 \
      [--max-frames 200]

Requirements
------------
  rubin_sim (maf), rubin_scheduler, healpy, astropy, h5py, lsst.resources
"""

import argparse
import io
import os
import sqlite3
import warnings
from datetime import timedelta

import h5py
import healpy as hp
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.ndimage import gaussian_filter
from scipy import ndimage
from astropy.coordinates import SkyCoord, EarthLocation, AltAz
from astropy.time import Time
import astropy.units as u

warnings.filterwarnings("ignore")

# ── optional LSST imports ─────────────────────────────────────────────────────
try:
    from lsst.resources import ResourcePath
    HAS_LSST_RESOURCES = True
except ImportError:
    HAS_LSST_RESOURCES = False
    print("WARNING: lsst.resources not found – using fallback HTTP fetch.")

try:
    import rubin_sim.maf as maf
    HAS_MAF = True
    print("rubin_sim.maf available – running full MAF suite.")
except ImportError:
    HAS_MAF = False
    print("WARNING: rubin_sim.maf not found – running lightweight metric equivalents.")

# ═══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ═══════════════════════════════════════════════════════════════════════════════
DEFAULT_CSV        = "feb5_data.csv"
DEFAULT_DB         = "baseline_v5.1.0_10yrs.db"
DEFAULT_DATE       = "2025-07-15"
OUTPUT_PNG         = "maf_scheduling_comparison.png"
OUTPUT_MAPS_PNG    = "maf_sky_maps.png"
OUTPUT_CSV         = "maf_results_summary.csv"

NSIDE              = 32
NEST               = True
RUBIN_LAT          = -30.244639
RUBIN_LON          = -70.749417
RUBIN_HEIGHT_M     = 2663.0
R_PROJECTION       = 10_000.0        # km
BIN_SIZE_KM        = 1_000
MIN_ALT_DEG        = 15.0
MAX_ZA_DEG         = 70.0

# Photometry
RUBIN_AREA         = np.pi * (6.4 / 2) ** 2   # m²
RUBIN_QE           = 0.8
EXPOSURE_TIME      = 30.0                       # s
READOUT_TIME       = 2.0                        # s
SLOT_TIME          = EXPOSURE_TIME + READOUT_TIME
PHOTON_FLUX_MAG20  = 100.0                      # photons/s/m² at zenith

# Slew model (Rubin simplified)
MAX_SLEW_ALT       = 3.5    # deg/s
MAX_SLEW_AZ        = 7.0    # deg/s
SLEW_SETTLE        = 2.0    # s

# MAF sky-coverage threshold
MIN_NVIS_FO        = 825    # fO benchmark: N visits over footprint

# Strategy colours / labels
STRATEGIES   = ("absolute", "motion", "scheduler")
STRAT_COLORS = {"absolute": "#2ca02c", "motion": "#1f77b4", "scheduler": "#d62728"}
STRAT_LABELS = {"absolute": "Abs Min (Green)",
                "motion":   "Motion Track (Blue)",
                "scheduler":"Scheduler (Red)"}

URL_COL      = "lsst.sal.DREAM.logevent_largeFileObjectAvailable.url"
TIME_COL     = "time"


# ═══════════════════════════════════════════════════════════════════════════════
# UTILITY HELPERS
# ═══════════════════════════════════════════════════════════════════════════════

def _location():
    return EarthLocation(lat=RUBIN_LAT*u.deg, lon=RUBIN_LON*u.deg,
                         height=RUBIN_HEIGHT_M*u.m)


def _ensure_time(t):
    if not isinstance(t, Time):
        t = Time(t)
    return t.utc


def transform_url(url):
    url = str(url).strip()
    if url.startswith("https://s3.cp.lsst.org/"):
        return url.replace("https://s3.cp.lsst.org/", "s3://lfa@")
    return url


def fetch_cloud_map(url):
    """Load cloud_sys HDF5 → (clouds, sigma) arrays, bad pixels → NaN."""
    if HAS_LSST_RESOURCES:
        rp = ResourcePath(url)
        with rp.open("rb") as fd:
            data = fd.read()
    else:
        import urllib.request
        with urllib.request.urlopen(url) as resp:
            data = resp.read()

    with h5py.File(io.BytesIO(data), "r") as f:
        clouds = np.array(f["clouds"],       dtype=float).ravel()
        sigma  = np.array(f["sigma"],        dtype=float).ravel()
        flags  = np.array(f["flags"],        dtype=int  ).ravel()
        vis    = np.array(f["mask_visible"], dtype=bool ).ravel()
        nobs   = np.array(f["nobs"],         dtype=int  ).ravel()

    bad = ~vis | (nobs == 0) | (flags > 0) | (sigma > 0.3) | ~np.isfinite(clouds)
    clouds[bad] = np.nan
    sigma[bad]  = np.nan
    return clouds, sigma


def healpix_to_grid(clouds, obstime):
    """Project HEALPix map onto flat-sky km grid (30×30 bins of BIN_SIZE_KM)."""
    npix = hp.nside2npix(NSIDE)
    pix  = np.arange(npix)
    th, ph = hp.pix2ang(NSIDE, pix, nest=NEST)
    ra  = np.degrees(ph)
    dec = 90.0 - np.degrees(th)

    t   = _ensure_time(obstime)
    loc = _location()
    sky = SkyCoord(ra=ra*u.deg, dec=dec*u.deg, frame="icrs")
    aa  = sky.transform_to(AltAz(obstime=t, location=loc))
    alt = aa.alt.deg
    az  = aa.az.deg % 360.0

    alt_r  = np.radians(alt)
    az_r   = np.radians(az)
    above  = alt > MIN_ALT_DEG
    scale  = np.where(above, R_PROJECTION / np.sin(alt_r), np.nan)
    xf     = -np.cos(alt_r) * np.sin(az_r) * scale
    yf     =  np.cos(alt_r) * np.cos(az_r) * scale
    vals   = np.where(above, clouds, np.nan)

    ok = ~np.isnan(vals) & (np.sqrt(xf**2 + yf**2) <= 15_000)
    x_edges = np.arange(-15_000, 15_001, BIN_SIZE_KM)
    y_edges = np.arange(-15_000, 15_001, BIN_SIZE_KM)

    Hs, _, _ = np.histogram2d(xf[ok], yf[ok], bins=[x_edges, y_edges], weights=vals[ok])
    Hc, _, _ = np.histogram2d(xf[ok], yf[ok], bins=[x_edges, y_edges])
    with np.errstate(divide="ignore", invalid="ignore"):
        H = Hs / Hc
    H[Hc == 0] = np.nan
    H = H.T
    xc = (x_edges[:-1] + x_edges[1:]) / 2
    yc = (y_edges[:-1] + y_edges[1:]) / 2
    Xg, Yg = np.meshgrid(xc, yc)
    H[np.sqrt(Xg**2 + Yg**2) > 15_000] = np.nan
    return H, x_edges, y_edges


def xy_to_altaz(x_km, y_km):
    r   = np.sqrt(x_km**2 + y_km**2)
    alt = float(np.degrees(np.arctan2(R_PROJECTION, r)))
    az  = float(np.degrees(np.arctan2(-x_km, y_km)) % 360.0)
    return alt, az


def altaz_to_radec(alt_deg, az_deg, obstime):
    t   = _ensure_time(obstime)
    loc = _location()
    aa  = SkyCoord(alt=alt_deg*u.deg, az=az_deg*u.deg,
                   frame=AltAz(obstime=t, location=loc))
    icrs = aa.icrs
    return float(icrs.ra.deg), float(icrs.dec.deg)


def grid_value(grid, x_km, y_km):
    xi = int(round(x_km / BIN_SIZE_KM + 15))
    yi = int(round(y_km / BIN_SIZE_KM + 15))
    if 0 <= xi < grid.shape[1] and 0 <= yi < grid.shape[0]:
        return grid[yi, xi]
    return np.nan


def find_abs_min(grid):
    if not np.any(~np.isnan(grid)):
        return 0, 0, np.nan
    idx     = np.nanargmin(grid)
    yi, xi  = np.unravel_index(idx, grid.shape)
    return (xi - 15) * BIN_SIZE_KM, (yi - 15) * BIN_SIZE_KM, grid[yi, xi]


def radec_to_xy(ra_deg, dec_deg, obstime):
    t   = _ensure_time(obstime)
    loc = _location()
    sky = SkyCoord(ra=ra_deg*u.deg, dec=dec_deg*u.deg, frame="icrs")
    aa  = sky.transform_to(AltAz(obstime=t, location=loc))
    alt, az = float(aa.alt.deg), float(aa.az.deg % 360.0)
    if alt < MIN_ALT_DEG:
        return None, None, alt, az
    alt_r, az_r = np.radians(alt), np.radians(az)
    scale = R_PROJECTION / np.sin(alt_r)
    x = -np.cos(alt_r) * np.sin(az_r) * scale
    y =  np.cos(alt_r) * np.cos(az_r) * scale
    return float(x), float(y), alt, az


def slew_time(alt1, az1, alt2, az2):
    da  = abs(alt2 - alt1)
    daz = abs(az2 - az1)
    if daz > 180:
        daz = 360 - daz
    return max(da / MAX_SLEW_ALT, daz / MAX_SLEW_AZ) + SLEW_SETTLE


def extinction_to_m5(ext_mag, airmass=1.2, base_m5=24.5):
    """Rough 5-sigma depth estimate from cloud extinction and airmass."""
    return base_m5 - ext_mag - 0.1 * (airmass - 1.0)


def photon_count(ext_mag):
    if np.isnan(ext_mag):
        return np.nan
    flux = 10 ** (-0.4 * ext_mag)
    return PHOTON_FLUX_MAG20 * RUBIN_AREA * RUBIN_QE * flux * EXPOSURE_TIME


def cloud_motion(g1, g2, sigma=5.0, search=10):
    g1s = gaussian_filter(np.nan_to_num(g1, nan=0), sigma)
    g2s = gaussian_filter(np.nan_to_num(g2, nan=0), sigma)
    m1  = ~np.isnan(g1)
    m2  = ~np.isnan(g2)
    best, bdx, bdy = -np.inf, 0, 0
    for dy in range(-search, search + 1):
        for dx in range(-search, search + 1):
            sh = ndimage.shift(g2s, (dy, dx), order=1, mode="constant", cval=0)
            sm = ndimage.shift(m2.astype(float), (dy, dx), order=0, mode="constant", cval=0) > 0.5
            v  = m1 & sm
            if v.sum() < 100:
                continue
            v1, v2 = g1s[v], sh[v]
            if np.std(v1) > 0 and np.std(v2) > 0:
                c = np.corrcoef(v1, v2)[0, 1]
                if c > best:
                    best, bdx, bdy = c, dx, dy
    return bdx, bdy, best


# ═══════════════════════════════════════════════════════════════════════════════
# DATA LOADING
# ═══════════════════════════════════════════════════════════════════════════════

def load_night_frames(csv_file, target_date):
    """Return cloud_sys HDF5 rows for the given observing night."""
    print(f"\nLoading DREAM CSV: {csv_file}")
    df = pd.read_csv(csv_file)
    df.columns = df.columns.str.replace('"', "").str.strip()
    df = df.dropna(subset=[URL_COL]).copy()
    df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors="coerce", utc=True)
    df = df.dropna(subset=[TIME_COL])
    df = df[df[URL_COL].str.contains(r"\.hdf5",    case=False, na=False, regex=True)]
    df = df[df[URL_COL].str.contains("cloud_sys",  case=False, na=False)]
    df = df.sort_values(TIME_COL).reset_index(drop=True)

    tgt   = pd.to_datetime(target_date).date()
    dates = df[TIME_COL].dt.date
    next_ = tgt + timedelta(days=1)
    mask  = (dates == tgt) | ((dates == next_) & (df[TIME_COL].dt.hour < 12))
    night = df[mask].reset_index(drop=True)
    print(f"  {len(night)} cloud_sys frames for night {target_date}")
    if len(night):
        print(f"  {night.iloc[0][TIME_COL]} → {night.iloc[-1][TIME_COL]}")
    return night


def load_scheduler_db(db_file):
    """Load one full night of scheduler pointings from an opsim SQLite file."""
    print(f"\nLoading scheduler DB: {db_file}")
    conn   = sqlite3.connect(db_file)
    cursor = conn.cursor()
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = [r[0] for r in cursor.fetchall()]
    table  = next((n for n in ["observations", "SummaryAllProps", "Summary", "obs"]
                   if n in tables), tables[0])
    obs    = pd.read_sql_query(f"SELECT * FROM {table}", conn)
    conn.close()

    # Normalise column names
    night_col = next((c for c in obs.columns if "night" in c.lower()), None)
    obs["night"] = obs[night_col].astype(int)
    for want, candidates in {
        "observationStartMJD": ["observationStartMJD", "observationstartmjd", "mjd"],
        "fieldRA":  ["fieldRA",  "fieldra",  "ra"],
        "fieldDec": ["fieldDec", "fielddec", "dec"],
        "filter":   ["filter",   "band"],
        "airmass":  ["airmass"],
        "seeingFwhmEff": ["seeingFwhmEff", "seeingfwhmeff", "seeing"],
        "fiveSigmaDepth": ["fiveSigmaDepth", "fivesigmadepth", "m5"],
        "slewTime":       ["slewTime", "slewtime"],
        "visitExposureTime": ["visitExposureTime", "visitexposuretime", "exptime"],
    }.items():
        if want not in obs.columns:
            for c in candidates:
                if c in obs.columns:
                    obs[want] = obs[c]
                    break

    # Select best night: most observations over at least 4 h
    scores = {}
    for n in obs["night"].unique()[:500]:
        sub = obs[obs["night"] == n]
        if len(sub) < 200:
            continue
        dur = (sub["observationStartMJD"].max() - sub["observationStartMJD"].min()) * 24
        if dur < 4:
            continue
        scores[n] = len(sub) * 10 + dur

    chosen = max(scores, key=scores.get) if scores else obs.groupby("night").size().idxmax()
    night  = obs[obs["night"] == chosen].sort_values("observationStartMJD").reset_index(drop=True)
    dur    = (night["observationStartMJD"].max() - night["observationStartMJD"].min()) * 24
    print(f"  Scheduler night {chosen}: {len(night)} obs, {dur:.1f} h")
    return night


def match_sched_to_frames(sched, frame_times):
    """Map each DREAM frame → nearest scheduler pointing by fractional night position."""
    n        = len(frame_times)
    f_fracs  = np.linspace(0, 1, n)
    mjd      = sched["observationStartMJD"].values
    rng      = mjd.max() - mjd.min() + 1e-12
    s_fracs  = (mjd - mjd.min()) / rng
    return [sched.iloc[np.argmin(np.abs(s_fracs - f))] for f in f_fracs]


# ═══════════════════════════════════════════════════════════════════════════════
# BUILD VISIT TABLES (synthetic opsim rows) for each strategy
# ═══════════════════════════════════════════════════════════════════════════════

def build_visit_tables(df_night, sched_obs, max_frames=None):
    """
    Load every DREAM frame, run three strategies, and build three visit DataFrames
    that mimic an opsim output table for input to MAF or lightweight metrics.

    Columns produced (matching opsim schema):
      observationStartMJD, fieldRA, fieldDec, filter, airmass,
      fiveSigmaDepth, seeingFwhmEff, slewTime, visitExposureTime,
      clouds_ext, photons, night
    """
    if max_frames:
        df_night = df_night.iloc[:max_frames]

    matched = match_sched_to_frames(sched_obs, df_night[TIME_COL].tolist())

    rows    = {s: [] for s in STRATEGIES}
    prev_alt = {s: 90.0 for s in STRATEGIES}
    prev_az  = {s:  0.0 for s in STRATEGIES}

    # Cloud motion state
    prev_grid   = None
    xp, yp      = 0.0, 0.0        # motion-tracking position
    HL          = 3                # history length for motion averaging

    dx_hist, dy_hist = [], []

    print(f"\nBuilding visit tables for {len(df_night)} frames …")
    for idx, (_, row) in enumerate(df_night.iterrows()):
        if idx % 50 == 0:
            print(f"  Frame {idx}/{len(df_night)}")

        url     = transform_url(row[URL_COL])
        t_visit = row[TIME_COL]

        try:
            clouds, _ = fetch_cloud_map(url)
        except Exception as e:
            print(f"    skip frame {idx}: {e}")
            prev_grid = None
            continue

        grid, _, _ = healpix_to_grid(clouds, t_visit)

        # ── MJD (approximate) ──────────────────────────────────────────
        t_astropy = _ensure_time(t_visit)
        mjd       = float(t_astropy.mjd)

        # ── Absolute minimum ───────────────────────────────────────────
        xa, ya, ext_a = find_abs_min(grid)
        alt_a, az_a   = xy_to_altaz(xa, ya)
        if ext_a is not np.nan and not np.isnan(ext_a) and alt_a > MIN_ALT_DEG:
            slew_a = slew_time(prev_alt["absolute"], prev_az["absolute"], alt_a, az_a)
            ra_a, dec_a = altaz_to_radec(alt_a, az_a, t_visit)
            am_a = 1.0 / np.sin(np.radians(alt_a))
            m5_a = extinction_to_m5(ext_a, am_a)
            rows["absolute"].append(dict(
                observationStartMJD = mjd,
                fieldRA = ra_a, fieldDec = dec_a,
                filter  = "r", airmass = am_a,
                fiveSigmaDepth = m5_a,
                seeingFwhmEff  = 0.7 + 0.1 * (am_a - 1),
                slewTime       = slew_a,
                visitExposureTime = EXPOSURE_TIME,
                clouds_ext     = ext_a,
                photons        = photon_count(ext_a),
                night          = 0,
            ))
            prev_alt["absolute"], prev_az["absolute"] = alt_a, az_a

        # ── Motion tracking ────────────────────────────────────────────
        if prev_grid is not None:
            dx, dy, conf = cloud_motion(prev_grid, grid)
            if conf > 0.5:
                dx_hist.append(dx); dy_hist.append(dy)
            if len(dx_hist) > HL:
                dx_hist.pop(0); dy_hist.pop(0)
            dx_avg = float(np.mean(dx_hist)) if dx_hist else 0.0
            dy_avg = float(np.mean(dy_hist)) if dy_hist else 0.0
            # Move opposite to cloud motion
            xp_new = xp - dx_avg * BIN_SIZE_KM
            yp_new = yp - dy_avg * BIN_SIZE_KM
            r_new  = np.sqrt(xp_new**2 + yp_new**2)
            if r_new > 14_000:
                xp_new *= 14_000 / r_new
                yp_new *= 14_000 / r_new
            val_new = grid_value(grid, xp_new, yp_new)
            if np.isnan(val_new) or val_new > 0.5:
                xp_new, yp_new, _ = find_abs_min(grid)
            xp, yp = xp_new, yp_new
        else:
            xp, yp, _ = find_abs_min(grid)

        ext_p = grid_value(grid, xp, yp)
        alt_p, az_p = xy_to_altaz(xp, yp)
        if not np.isnan(ext_p) and alt_p > MIN_ALT_DEG:
            slew_p = slew_time(prev_alt["motion"], prev_az["motion"], alt_p, az_p)
            ra_p, dec_p = altaz_to_radec(alt_p, az_p, t_visit)
            am_p = 1.0 / np.sin(np.radians(alt_p))
            m5_p = extinction_to_m5(ext_p, am_p)
            rows["motion"].append(dict(
                observationStartMJD = mjd,
                fieldRA = ra_p, fieldDec = dec_p,
                filter  = "r", airmass = am_p,
                fiveSigmaDepth = m5_p,
                seeingFwhmEff  = 0.7 + 0.1 * (am_p - 1),
                slewTime       = slew_p,
                visitExposureTime = EXPOSURE_TIME,
                clouds_ext     = ext_p,
                photons        = photon_count(ext_p),
                night          = 0,
            ))
            prev_alt["motion"], prev_az["motion"] = alt_p, az_p

        prev_grid = grid.copy()

        # ── Scheduler ──────────────────────────────────────────────────
        obs     = matched[idx]
        xs, ys, alt_s, az_s = radec_to_xy(
            float(obs["fieldRA"]), float(obs["fieldDec"]), t_visit)
        if xs is not None:
            ext_s = grid_value(grid, xs, ys)
            if not np.isnan(ext_s) and alt_s > MIN_ALT_DEG:
                slew_s = slew_time(prev_alt["scheduler"], prev_az["scheduler"], alt_s, az_s)
                am_s   = 1.0 / np.sin(np.radians(alt_s))
                m5_s   = extinction_to_m5(ext_s, am_s)
                # Inherit realistic seeing/depth from opsim if available
                raw_m5 = obs.get("fiveSigmaDepth", m5_s)
                if pd.isna(raw_m5):
                    raw_m5 = m5_s
                rows["scheduler"].append(dict(
                    observationStartMJD = mjd,
                    fieldRA  = float(obs["fieldRA"]),
                    fieldDec = float(obs["fieldDec"]),
                    filter   = str(obs.get("filter", "r")).strip("'"),
                    airmass  = am_s,
                    fiveSigmaDepth  = float(raw_m5) - ext_s,   # apply cloud penalty
                    seeingFwhmEff   = float(obs.get("seeingFwhmEff", 0.8) or 0.8),
                    slewTime        = slew_s,
                    visitExposureTime = EXPOSURE_TIME,
                    clouds_ext      = ext_s,
                    photons         = photon_count(ext_s),
                    night           = 0,
                ))
                prev_alt["scheduler"], prev_az["scheduler"] = alt_s, az_s

    dfs = {s: pd.DataFrame(rows[s]) for s in STRATEGIES}
    for s, df in dfs.items():
        print(f"  {s:10s}: {len(df):4d} visits")
    return dfs


# ═══════════════════════════════════════════════════════════════════════════════
# LIGHTWEIGHT METRICS (no rubin_sim.maf required)
# ═══════════════════════════════════════════════════════════════════════════════

def lightweight_metrics(dfs):
    """
    Compute a comprehensive set of per-strategy scalar metrics without MAF.
    Returns a dict of {metric_name: {strategy: value}}.
    """
    results = {}

    for s, df in dfs.items():
        if df.empty:
            continue
        mjd  = df["observationStartMJD"].values
        sort_idx = np.argsort(mjd)
        mjd  = mjd[sort_idx]
        ph   = df["photons"].values[sort_idx]
        m5   = df["fiveSigmaDepth"].values[sort_idx]
        am   = df["airmass"].values[sort_idx]
        see  = df["seeingFwhmEff"].values[sort_idx]
        slew = df["slewTime"].values[sort_idx]
        ext  = df["clouds_ext"].values[sort_idx]

        # ── Count / efficiency ───────────────────────────────────────
        results.setdefault("N_visits",         {})[s] = len(df)
        results.setdefault("Total_photons",    {})[s] = float(np.nansum(ph))

        dur_h = (mjd[-1] - mjd[0]) * 24 if len(mjd) > 1 else 0.0
        total_slew_h = float(np.sum(slew)) / 3600.0
        results.setdefault("Night_duration_h", {})[s] = dur_h
        results.setdefault("Total_slew_h",     {})[s] = total_slew_h
        exp_h = len(df) * EXPOSURE_TIME / 3600.0
        results.setdefault("Total_exposure_h", {})[s] = exp_h
        open_frac = exp_h / (dur_h + 1e-12)
        results.setdefault("Open_shutter_frac",{})[s] = float(open_frac)

        # ── Airmass ──────────────────────────────────────────────────
        results.setdefault("Median_airmass",   {})[s] = float(np.nanmedian(am))
        results.setdefault("Max_airmass",      {})[s] = float(np.nanmax(am))
        results.setdefault("Mean_airmass",     {})[s] = float(np.nanmean(am))

        # ── Seeing ───────────────────────────────────────────────────
        results.setdefault("Median_seeing",    {})[s] = float(np.nanmedian(see))

        # ── Depth ────────────────────────────────────────────────────
        results.setdefault("Median_m5",        {})[s] = float(np.nanmedian(m5))
        results.setdefault("Mean_m5",          {})[s] = float(np.nanmean(m5))
        results.setdefault("Coadd_m5_proxy",   {})[s] = float(
            -2.5 * np.log10(np.nansum(10 ** (-0.4 * m5 * 2))) / 2
            if len(m5) else np.nan)

        # ── Cloud extinction ─────────────────────────────────────────
        results.setdefault("Mean_cloud_ext",   {})[s] = float(np.nanmean(ext))
        results.setdefault("Median_cloud_ext", {})[s] = float(np.nanmedian(ext))
        results.setdefault("Min_cloud_ext",    {})[s] = float(np.nanmin(ext))

        # ── Cadence / gaps ───────────────────────────────────────────
        if len(mjd) > 1:
            gaps = np.diff(mjd) * 24 * 60   # minutes
            results.setdefault("Max_gap_min",   {})[s] = float(gaps.max())
            results.setdefault("Median_gap_min",{})[s] = float(np.median(gaps))
        else:
            results.setdefault("Max_gap_min",   {})[s] = np.nan
            results.setdefault("Median_gap_min",{})[s] = np.nan

        # ── Slew ─────────────────────────────────────────────────────
        results.setdefault("Mean_slew_s",      {})[s] = float(np.nanmean(slew))
        results.setdefault("Median_slew_s",    {})[s] = float(np.nanmedian(slew))
        results.setdefault("Max_slew_s",       {})[s] = float(np.nanmax(slew))

        # ── Sky coverage (healpix) ────────────────────────────────────
        pix_set = set()
        for _, r in df.iterrows():
            th  = np.radians(90.0 - r["fieldDec"])
            phi = np.radians(r["fieldRA"] % 360.0)
            pix_set.add(hp.ang2pix(NSIDE, th, phi, nest=NEST))
        sky_frac = len(pix_set) / hp.nside2npix(NSIDE)
        results.setdefault("Sky_frac_visited", {})[s] = float(sky_frac)

        # ── Uniformity (Kuiper-style spatial spread) ─────────────────
        ra_vals  = np.radians(df["fieldRA"].values)
        dec_vals = df["fieldDec"].values
        cos_dec  = np.cos(np.radians(dec_vals))
        # Normalised spatial entropy over visited pixels
        counts   = np.zeros(hp.nside2npix(NSIDE))
        for p in pix_set:
            counts[p] = 1
        frac_vis = len(pix_set) / hp.nside2npix(NSIDE)
        results.setdefault("Uniformity_frac", {})[s] = float(frac_vis)

        # ── SRD-style fO proxy ────────────────────────────────────────
        # fraction of healpix pixels with ≥1 visit
        results.setdefault("fO_area_proxy",    {})[s] = float(len(pix_set) * hp.nside2pixarea(NSIDE, degrees=True))

        # ── Survey depth relative to photometric ─────────────────────
        phot_m5 = extinction_to_m5(0.0)    # extinction=0, baseline
        delta_depth = np.nanmean(m5) - phot_m5
        results.setdefault("Delta_depth_vs_phot", {})[s] = float(delta_depth)

    return results


# ═══════════════════════════════════════════════════════════════════════════════
# FULL MAF METRICS (when rubin_sim.maf available)
# ═══════════════════════════════════════════════════════════════════════════════

def run_maf_metrics(dfs, out_dir="maf_outputs"):
    """
    Run a full MAF metric suite over an in-memory opsim-like visit table.
    Uses rubin_sim.maf with a temporary SQLite database written from each
    strategy's visit DataFrame.
    """
    import tempfile, os
    os.makedirs(out_dir, exist_ok=True)

    # Columns required by MAF
    REQUIRED = [
        "observationId", "fieldRA", "fieldDec", "filter",
        "observationStartMJD", "visitExposureTime",
        "airmass", "seeingFwhmEff", "fiveSigmaDepth",
        "slewTime", "night",
        # MAF internals (can be zero)
        "rotSkyPos", "numExposures", "skyBrightness",
        "slewDistance", "altitude", "azimuth",
        "moonAlt", "sunAlt", "cloud",
    ]

    maf_results = {}

    for s, df in dfs.items():
        if df.empty:
            maf_results[s] = {}
            continue

        print(f"\n  Running MAF on {s} ({len(df)} visits) …")

        # ── Prepare full visit table ───────────────────────────────────
        full = df.copy()
        full["observationId"]   = np.arange(len(full), dtype=int)
        full["rotSkyPos"]       = 0.0
        full["numExposures"]    = 1
        full["skyBrightness"]   = 21.0
        full["slewDistance"]    = full["slewTime"] * 3.5  # rough deg
        full["altitude"]        = np.degrees(np.arcsin(1.0 / full["airmass"].clip(1, 5)))
        full["azimuth"]         = 180.0
        full["moonAlt"]         = -30.0
        full["sunAlt"]          = -18.0
        full["cloud"]           = full["clouds_ext"].fillna(0.5)
        for col in REQUIRED:
            if col not in full.columns:
                full[col] = 0.0

        # ── Write temp SQLite ──────────────────────────────────────────
        with tempfile.NamedTemporaryFile(suffix=".db", delete=False) as tmp:
            db_path = tmp.name

        conn = sqlite3.connect(db_path)
        full[REQUIRED].to_sql("observations", conn, if_exists="replace", index=False)
        conn.close()

        strat_out = os.path.join(out_dir, s)
        os.makedirs(strat_out, exist_ok=True)

        # ── Define metric bundles ──────────────────────────────────────
        uni   = maf.UniSlicer()
        healpix = maf.HealpixSlicer(nside=NSIDE)

        bundles = [
            # Count
            maf.MetricBundle(maf.CountMetric("observationId"),      uni, "", run_name=s),
            # Airmass distribution
            maf.MetricBundle(maf.MedianMetric("airmass"),            uni, "", run_name=s),
            maf.MetricBundle(maf.MaxMetric("airmass"),               uni, "", run_name=s),
            maf.MetricBundle(maf.MeanMetric("airmass"),              uni, "", run_name=s),
            # Seeing
            maf.MetricBundle(maf.MedianMetric("seeingFwhmEff"),      uni, "", run_name=s),
            # 5-sigma depth
            maf.MetricBundle(maf.MedianMetric("fiveSigmaDepth"),     uni, "", run_name=s),
            maf.MetricBundle(maf.MeanMetric("fiveSigmaDepth"),       uni, "", run_name=s),
            maf.MetricBundle(maf.MinMetric("fiveSigmaDepth"),        uni, "", run_name=s),
            # Slew
            maf.MetricBundle(maf.MeanMetric("slewTime"),             uni, "", run_name=s),
            maf.MetricBundle(maf.SumMetric("slewTime"),              uni, "", run_name=s),
            # Open-shutter fraction
            maf.MetricBundle(maf.OpenShutterFractionMetric(),        uni, "", run_name=s),
            # Gaps
            maf.MetricBundle(maf.MaxGapMetric(),                     uni, "", run_name=s),
            # Sky-map depth
            maf.MetricBundle(maf.Coaddm5Metric(),                healpix, "", run_name=s),
            # fO area
            maf.MetricBundle(
                maf.fOArea(col="observationId", nside=NSIDE),   healpix, "", run_name=s),
        ]

        bg = maf.MetricBundleGroup(bundles, db_path, out_dir=strat_out)
        bg.run_all()
        os.unlink(db_path)

        # Harvest scalar results
        scalars = {}
        for b in bundles:
            mv = b.metric_values
            if hasattr(mv, "compressed") and mv.compressed().size == 1:
                scalars[b.metric.name] = float(mv.compressed()[0])
            elif hasattr(mv, "data") and mv.data.size == 1:
                scalars[b.metric.name] = float(mv.data.ravel()[0])
        maf_results[s] = scalars

    return maf_results


# ═══════════════════════════════════════════════════════════════════════════════
# PRINT COMPARISON TABLE
# ═══════════════════════════════════════════════════════════════════════════════

def print_table(results, title="METRIC COMPARISON"):
    print("\n" + "═" * 72)
    print(f"  {title}")
    print("═" * 72)
    col_w = 20
    strat_w = 18

    hdr = f"{'Metric':<{col_w}}" + "".join(f"{STRAT_LABELS[s][:strat_w]:<{strat_w}}"
                                              for s in STRATEGIES)
    print(hdr)
    print("─" * len(hdr))

    for metric, vals in sorted(results.items()):
        row = f"{metric:<{col_w}}"
        best_val = None
        best_fn  = min   # lower is better by default
        if any(k in metric for k in ("photon", "N_vis", "m5", "exposure",
                                      "frac", "sky", "fO", "depth", "open")):
            best_fn = max
        try:
            numeric_vals = {k: v for k, v in vals.items() if not np.isnan(v)}
            if numeric_vals:
                best_val = best_fn(numeric_vals.values())
        except Exception:
            pass

        for s in STRATEGIES:
            v = vals.get(s, np.nan)
            fmt = f"{v:.4f}" if not np.isnan(v) else "  N/A  "
            marker = " ★" if (best_val is not None and
                              not np.isnan(v) and
                              abs(v - best_val) < 1e-9) else "  "
            row += f"{fmt}{marker}".ljust(strat_w)
        print(row)
    print("═" * 72)
    print("  ★ = best value for that metric")


# ═══════════════════════════════════════════════════════════════════════════════
# PLOTS
# ═══════════════════════════════════════════════════════════════════════════════

def plot_summary(results, dfs, target_date, output=OUTPUT_PNG):
    """12-panel summary figure comparing all three strategies."""
    print(f"\nGenerating summary plot → {output}")

    fig = plt.figure(figsize=(22, 18))
    fig.suptitle(
        f"MAF-Style Metric Comparison — Night {target_date}\n"
        "DREAM Cloud Scheduling Strategies",
        fontsize=15, weight="bold", y=0.98)

    gs = gridspec.GridSpec(4, 3, figure=fig, hspace=0.45, wspace=0.35)

    colors = [STRAT_COLORS[s] for s in STRATEGIES]
    labels = [STRAT_LABELS[s]  for s in STRATEGIES]

    def bar_panel(ax, metric_key, ylabel, title, higher_better=True, scale=1.0):
        vals   = [results.get(metric_key, {}).get(s, np.nan) * scale for s in STRATEGIES]
        finite = [v for v in vals if not np.isnan(v)]
        if not finite:
            ax.set_title(title + " (no data)"); return
        bars = ax.bar(labels, vals, color=colors, alpha=0.78,
                      edgecolor="black", linewidth=1.2)
        best = max(finite) if higher_better else min(finite)
        for bar, val in zip(bars, vals):
            if np.isnan(val):
                continue
            ax.text(bar.get_x() + bar.get_width()/2, val,
                    f"{val:.3f}", ha="center", va="bottom", fontsize=8.5, weight="bold")
            if abs(val - best) < 1e-9:
                bar.set_edgecolor("gold")
                bar.set_linewidth(3)
        ax.set_ylabel(ylabel, fontsize=9)
        ax.set_title(title, fontsize=10, weight="bold")
        ax.grid(axis="y", alpha=0.3)
        ax.tick_params(axis="x", labelsize=7.5, rotation=15)

    # Row 0
    bar_panel(fig.add_subplot(gs[0, 0]), "N_visits",         "Count",     "N Visits",          True)
    bar_panel(fig.add_subplot(gs[0, 1]), "Total_photons",    "Photons",   "Total Photons",     True, 1e-6)
    bar_panel(fig.add_subplot(gs[0, 2]), "Open_shutter_frac","Fraction",  "Open-Shutter Frac", True)

    # Row 1
    bar_panel(fig.add_subplot(gs[1, 0]), "Median_m5",        "mag",       "Median 5σ Depth",   True)
    bar_panel(fig.add_subplot(gs[1, 1]), "Median_airmass",   "Airmass",   "Median Airmass",    False)
    bar_panel(fig.add_subplot(gs[1, 2]), "Mean_cloud_ext",   "mag",       "Mean Cloud Extinction", False)

    # Row 2
    bar_panel(fig.add_subplot(gs[2, 0]), "Mean_slew_s",      "s",         "Mean Slew Time",    False)
    bar_panel(fig.add_subplot(gs[2, 1]), "Total_slew_h",     "hours",     "Total Slew Time",   False)
    bar_panel(fig.add_subplot(gs[2, 2]), "fO_area_proxy",    "deg²",      "Sky Area Visited",  True)

    # Row 3: time-series photons & m5 histograms
    ax_ts  = fig.add_subplot(gs[3, :2])
    ax_m5h = fig.add_subplot(gs[3, 2])

    for s in STRATEGIES:
        df = dfs[s]
        if df.empty:
            continue
        mjd = df["observationStartMJD"].values
        ph  = df["photons"].values
        t_h = (mjd - mjd.min()) * 24
        ax_ts.plot(t_h, ph, color=STRAT_COLORS[s], alpha=0.6, lw=1.2,
                   label=STRAT_LABELS[s])

        m5  = df["fiveSigmaDepth"].dropna()
        ax_m5h.hist(m5, bins=25, color=STRAT_COLORS[s], alpha=0.5,
                    label=STRAT_LABELS[s], histtype="stepfilled", edgecolor="black", lw=0.8)

    ax_ts.set_xlabel("Hours into night", fontsize=9)
    ax_ts.set_ylabel("Photons / 30 s exposure", fontsize=9)
    ax_ts.set_title("Photon Count Per Exposure Over Night", fontsize=10, weight="bold")
    ax_ts.legend(fontsize=7); ax_ts.grid(alpha=0.3)
    ax_ts.ticklabel_format(axis="y", style="scientific", scilimits=(0,0))

    ax_m5h.set_xlabel("5σ Limiting Magnitude", fontsize=9)
    ax_m5h.set_ylabel("Count", fontsize=9)
    ax_m5h.set_title("Survey Depth Distribution", fontsize=10, weight="bold")
    ax_m5h.legend(fontsize=7); ax_m5h.grid(alpha=0.3)

    plt.savefig(output, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  → {output}")


def plot_sky_maps(dfs, target_date, output=OUTPUT_MAPS_PNG):
    """
    Sky maps of coadded 5σ depth drawn with pure matplotlib (no healpy figure magic).
    Uses a simple equirectangular projection: RA on x-axis, Dec on y-axis.
    """
    print(f"\nGenerating sky maps → {output}")

    fig, axes = plt.subplots(1, 3, figsize=(21, 5),
                             subplot_kw={"projection": "mollweide"})
    fig.suptitle(f"Coadded 5σ Depth Sky Maps  —  Night {target_date}",
                 fontsize=13, weight="bold", y=1.01)

    vmin, vmax = 22.0, 26.0

    for ax, s in zip(axes, STRATEGIES):
        df = dfs[s]

        # Collect (RA, Dec, m5) per visit
        ras, decs, m5s = [], [], []
        if not df.empty:
            for _, r in df.iterrows():
                if np.isfinite(r["fiveSigmaDepth"]):
                    ras.append(r["fieldRA"])
                    decs.append(r["fieldDec"])
                    m5s.append(r["fiveSigmaDepth"])

        ax.set_title(STRAT_LABELS[s], fontsize=10, weight="bold", pad=6)
        ax.grid(True, alpha=0.3, lw=0.5)
        ax.set_xlabel("RA",  fontsize=8)
        ax.set_ylabel("Dec", fontsize=8)
        ax.tick_params(labelsize=7)

        if ras:
            # mollweide wants radians: RA shifted to [-π, π]
            ra_rad  = np.radians(np.array(ras))
            ra_rad  = ((ra_rad + np.pi) % (2 * np.pi)) - np.pi   # wrap to [-π,π]
            dec_rad = np.radians(np.array(decs))
            m5_arr  = np.array(m5s)

            sc = ax.scatter(ra_rad, dec_rad, c=m5_arr,
                            cmap="viridis", vmin=vmin, vmax=vmax,
                            s=6, alpha=0.8, lw=0)
        else:
            # Empty — draw placeholder scatter so colorbar still works
            sc = ax.scatter([], [], c=[], cmap="viridis", vmin=vmin, vmax=vmax, s=6)
            ax.text(0, 0, "No data", ha="center", va="center",
                    fontsize=10, color="gray", transform=ax.transData)

        plt.colorbar(sc, ax=ax, orientation="horizontal", pad=0.06,
                     fraction=0.046, label="coadd 5σ (mag)")

    plt.tight_layout()
    plt.savefig(output, dpi=130, bbox_inches="tight")
    plt.close()
    print(f"  → {output}")


def plot_maf_results(maf_results, output="maf_native_metrics.png"):
    """Bar chart of native MAF scalar metrics."""
    all_metrics = sorted(set(k for v in maf_results.values() for k in v))
    n = len(all_metrics)
    if n == 0:
        return

    ncols = 3
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(18, 4 * nrows))
    fig.suptitle("Native MAF Metrics by Strategy", fontsize=14, weight="bold")
    axes = axes.ravel()

    colors = [STRAT_COLORS[s] for s in STRATEGIES]
    labels = [STRAT_LABELS[s]  for s in STRATEGIES]

    for i, metric in enumerate(all_metrics):
        ax   = axes[i]
        vals = [maf_results.get(s, {}).get(metric, np.nan) for s in STRATEGIES]
        bars = ax.bar(labels, vals, color=colors, alpha=0.75,
                      edgecolor="black", linewidth=1.1)
        for bar, v in zip(bars, vals):
            if not np.isnan(v):
                ax.text(bar.get_x() + bar.get_width()/2, v,
                        f"{v:.3g}", ha="center", va="bottom", fontsize=8)
        ax.set_title(metric[:40], fontsize=9, weight="bold")
        ax.grid(axis="y", alpha=0.3)
        ax.tick_params(axis="x", labelsize=7, rotation=20)

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    plt.tight_layout()
    plt.savefig(output, dpi=130, bbox_inches="tight")
    plt.close()
    print(f"  MAF native plot → {output}")


# ═══════════════════════════════════════════════════════════════════════════════
# MAIN
# ═══════════════════════════════════════════════════════════════════════════════

def main(
    csv=DEFAULT_CSV,
    db=DEFAULT_DB,
    date=DEFAULT_DATE,
    max_frames=None,
    out_dir=".",
    _args=None,           # internal: pre-parsed namespace (overrides all)
):
    """
    Entry point — callable directly from Python/Jupyter or from the command line.

    Jupyter / IPython usage
    -----------------------
        from maf_scheduling_comparison import main
        main(csv="feb5_data.csv", db="baseline_v5.1.0_10yrs.db",
             date="2025-07-15", max_frames=200)

    Command-line usage
    ------------------
        python maf_scheduling_comparison.py \\
            --csv feb5_data.csv --db baseline_v5.1.0_10yrs.db \\
            --date 2025-07-15 --max-frames 200
    """
    # ── Argument resolution ───────────────────────────────────────────────────
    # When called from the CLI, _args is populated by _cli_main().
    # When called directly (Jupyter / import), keyword args are used as-is.
    if _args is not None:
        csv        = _args.csv
        db         = _args.db
        date       = _args.date
        max_frames = _args.max_frames
        out_dir    = _args.out_dir

    # Convenience: expose as a simple namespace so the rest of the function
    # uses attribute access uniformly.
    class _A:
        pass
    args = _A()
    args.csv        = csv
    args.db         = db
    args.date       = date
    args.max_frames = max_frames
    args.out_dir    = out_dir

    os.makedirs(args.out_dir, exist_ok=True)
    png_main  = os.path.join(args.out_dir, OUTPUT_PNG)
    png_maps  = os.path.join(args.out_dir, OUTPUT_MAPS_PNG)
    csv_out   = os.path.join(args.out_dir, OUTPUT_CSV)
    png_maf   = os.path.join(args.out_dir, "maf_native_metrics.png")

    print("\n" + "═" * 72)
    print("  MAF METRICS ON DREAM CLOUD SCHEDULING STRATEGIES")
    print(f"  Night: {args.date}")
    print("═" * 72)

    # 1. Load data
    df_night  = load_night_frames(args.csv, args.date)
    if df_night.empty:
        print(f"ERROR: No DREAM frames found for {args.date}.  Exiting.")
        return

    sched_obs = load_scheduler_db(args.db)

    # 2. Build synthetic visit tables
    dfs = build_visit_tables(df_night, sched_obs, max_frames=args.max_frames)

    # 3. Run lightweight metrics (always available)
    print("\n" + "═" * 50)
    print("  Running lightweight metrics …")
    lw_results = lightweight_metrics(dfs)
    print_table(lw_results, "LIGHTWEIGHT METRIC COMPARISON")

    # 4. Run full MAF metrics if available
    maf_results = {}
    if HAS_MAF:
        print("\n" + "═" * 50)
        print("  Running native MAF metrics …")
        try:
            maf_results = run_maf_metrics(dfs, out_dir=os.path.join(args.out_dir, "maf_outputs"))
            print_table(maf_results, "NATIVE MAF METRIC COMPARISON")
            plot_maf_results(maf_results, output=png_maf)
        except Exception as e:
            print(f"  MAF run failed: {e}")
            import traceback; traceback.print_exc()
    else:
        print("\n  (skipping native MAF – rubin_sim.maf not installed)")

    # 5. Combine results
    combined = {**lw_results, **maf_results}

    # 6. Save CSV
    rows = []
    for metric, vals in sorted(combined.items()):
        row = {"metric": metric}
        row.update(vals)
        rows.append(row)
    pd.DataFrame(rows).to_csv(csv_out, index=False)
    print(f"\n  Results saved → {csv_out}")

    # 7. Plots
    plot_summary(lw_results, dfs, args.date, output=png_main)
    try:
        plot_sky_maps(dfs, args.date, output=png_maps)
    except Exception as e:
        import traceback
        print(f"  Sky map failed: {e}")
        traceback.print_exc()

    # 8. Final scorecard
    print("\n" + "═" * 72)
    print("  FINAL SCORECARD  (best strategy per metric)")
    print("═" * 72)
    wins = {s: 0 for s in STRATEGIES}
    for metric, vals in lw_results.items():
        finite = {k: v for k, v in vals.items() if not np.isnan(v)}
        if not finite:
            continue
        higher_better = any(k in metric for k in (
            "photon", "N_vis", "m5", "exposure", "frac", "sky", "fO",
            "depth", "open", "Uniformity"))
        best_fn = max if higher_better else min
        winner  = best_fn(finite, key=finite.get)
        wins[winner] += 1
        print(f"  {metric:<28s}  → {STRAT_LABELS[winner]}")

    print("\n  Wins per strategy:")
    for s in STRATEGIES:
        print(f"    {STRAT_LABELS[s]:<30s}: {wins[s]:2d}")
    print("═" * 72)
    print("\nDone.")



def _cli_main():
    """CLI entry-point — strips Jupyter kernel args before parsing."""
    import sys
    parser = argparse.ArgumentParser(
        description="MAF metrics on DREAM cloud scheduling strategies")
    parser.add_argument("--csv",        default=DEFAULT_CSV)
    parser.add_argument("--db",         default=DEFAULT_DB)
    parser.add_argument("--date",       default=DEFAULT_DATE)
    parser.add_argument("--max-frames", type=int, default=None, dest="max_frames")
    parser.add_argument("--out-dir",    default=".",            dest="out_dir")

    # Strip unknown flags injected by Jupyter (e.g. -f <kernel-json>)
    known_flags = {"--csv", "--db", "--date", "--max-frames", "--out-dir", "-h", "--help"}
    clean, skip = [], False
    for tok in sys.argv[1:]:
        if skip:
            skip = False
            continue
        if tok in known_flags or any(tok.startswith(f + "=") for f in known_flags):
            clean.append(tok)
        elif tok.startswith("-") and tok not in known_flags:
            skip = True   # drop this flag AND its value
        else:
            clean.append(tok)

    main(_args=parser.parse_args(clean))


if __name__ == "__main__":
    _cli_main()

rubin_sim.maf available – running full MAF suite.

════════════════════════════════════════════════════════════════════════
  MAF METRICS ON DREAM CLOUD SCHEDULING STRATEGIES
  Night: 2025-07-15
════════════════════════════════════════════════════════════════════════

Loading DREAM CSV: feb5_data.csv
  2529 cloud_sys frames for night 2025-07-15
  2025-07-15 00:00:12.958000+00:00 → 2025-07-16 10:52:31.292000+00:00

Loading scheduler DB: baseline_v5.1.0_10yrs.db
  Scheduler night 572: 1139 obs, 11.7 h

Building visit tables for 2529 frames …
  Frame 0/2529
  Frame 50/2529
  Frame 100/2529
  Frame 150/2529
  Frame 200/2529
  Frame 250/2529
  Frame 300/2529
  Frame 350/2529
  Frame 400/2529
  Frame 450/2529
  Frame 500/2529
  Frame 550/2529
  Frame 600/2529
  Frame 650/2529
  Frame 700/2529
  Frame 750/2529
  Frame 800/2529
  Frame 850/2529
  Frame 900/2529
  Frame 950/2529
  Frame 1000/2529
  Frame 1050/2529
  Frame 1100/2529
  Frame 1150/2529
  Frame 1200/2529
  Frame 1250/2529
  Frame 1

Traceback (most recent call last):
  File "/tmp/ipykernel_4310/1133125980.py", line 1076, in main
    maf_results = run_maf_metrics(dfs, out_dir=os.path.join(args.out_dir, "maf_outputs"))
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_4310/1133125980.py", line 748, in run_maf_metrics
    maf.fOArea(col="observationId", nside=NSIDE),   healpix, "", run_name=s),
    ^^^^^^^^^^
AttributeError: module 'rubin_sim.maf' has no attribute 'fOArea'. Did you mean: 'FOArea'?


  → ./maf_scheduling_comparison.png

Generating sky maps → ./maf_sky_maps.png
  → ./maf_sky_maps.png

════════════════════════════════════════════════════════════════════════
  FINAL SCORECARD  (best strategy per metric)
════════════════════════════════════════════════════════════════════════
  N_visits                      → Abs Min (Green)
  Total_photons                 → Abs Min (Green)
  Night_duration_h              → Scheduler (Red)
  Total_slew_h                  → Scheduler (Red)
  Total_exposure_h              → Abs Min (Green)
  Open_shutter_frac             → Abs Min (Green)
  Median_airmass                → Scheduler (Red)
  Max_airmass                   → Scheduler (Red)
  Mean_airmass                  → Scheduler (Red)
  Median_seeing                 → Motion Track (Blue)
  Median_m5                     → Abs Min (Green)
  Mean_m5                       → Abs Min (Green)
  Coadd_m5_proxy                → Abs Min (Green)
  Mean_cloud_ext                → Abs Min (Green)
  

In [1]:
#!/usr/bin/env python3
"""
MAF Metrics Applied to DREAM Cloud Scheduling Strategies — ALL NIGHTS
======================================================================
Iterates over every partially-cloudy night in the DREAM CSV, runs a full
MAF-style metric suite for three pointing strategies, then produces a
cross-night summary.

Three strategies per night
--------------------------
  Green  – Absolute minimum cloud extinction
  Blue   – Cloud motion tracking prediction
  Red    – Baseline scheduler (opsim .db file)

Night selection
---------------
Only nights that pass the patchiness filter are analysed:
  • valid_frac between PATCHINESS_LOW_FRAC and PATCHINESS_HIGH_FRAC
  • spatial variance  ≥ PATCHINESS_VAR_THRESH

Per-night outputs (saved under --out-dir/<YYYYMMDD>/)
------------------------------------------------------
  metrics_<date>.png          – 12-panel MAF-style bar/timeline chart
  sky_maps_<date>.png         – Mollweide depth maps per strategy
  maf_results_<date>.csv      – raw metric values

Cross-night summary outputs (saved under --out-dir/)
-----------------------------------------------------
  all_nights_summary.csv      – one row per night × strategy
  cross_night_comparison.png  – multi-panel time-series + bar charts

Usage
-----
  python maf_multinight_scheduling.py \\
      --csv   feb5_data.csv \\
      --db    baseline_v5.1.0_10yrs.db \\
      [--max-frames 200]   \\
      [--out-dir  results/]

Requirements
------------
  rubin_sim (maf), rubin_scheduler, healpy, astropy, h5py, lsst.resources
  (falls back gracefully if rubin_sim.maf is absent)
"""

import argparse
import io
import os
import sqlite3
import warnings
from datetime import timedelta, date

import h5py
import healpy as hp
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.ndimage import gaussian_filter
from scipy import ndimage
from astropy.coordinates import SkyCoord, EarthLocation, AltAz
from astropy.time import Time
import astropy.units as u

warnings.filterwarnings("ignore")

# ── optional LSST imports ─────────────────────────────────────────────────────
try:
    from lsst.resources import ResourcePath
    HAS_LSST_RESOURCES = True
except ImportError:
    HAS_LSST_RESOURCES = False
    print("WARNING: lsst.resources not found – using fallback HTTP fetch.")

try:
    import rubin_sim.maf as maf
    HAS_MAF = True
    print("rubin_sim.maf available – running full MAF suite.")
except ImportError:
    HAS_MAF = False
    print("WARNING: rubin_sim.maf not found – running lightweight metric equivalents.")


# ═══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ═══════════════════════════════════════════════════════════════════════════════
DEFAULT_CSV   = "feb5_data.csv"
DEFAULT_DB    = "baseline_v5.1.0_10yrs.db"
DEFAULT_OUTDIR = "maf_multinight_results"

NSIDE              = 32
NEST               = True
RUBIN_LAT          = -30.244639
RUBIN_LON          = -70.749417
RUBIN_HEIGHT_M     = 2663.0
R_PROJECTION       = 10_000.0        # km
BIN_SIZE_KM        = 1_000
MIN_ALT_DEG        = 15.0

# Photometry
RUBIN_AREA         = np.pi * (6.4 / 2) ** 2
RUBIN_QE           = 0.8
EXPOSURE_TIME      = 30.0
READOUT_TIME       = 2.0
SLOT_TIME          = EXPOSURE_TIME + READOUT_TIME
PHOTON_FLUX_MAG20  = 100.0

# Slew model
MAX_SLEW_ALT       = 3.5
MAX_SLEW_AZ        = 7.0
SLEW_SETTLE        = 2.0

# Patchiness thresholds
PATCHINESS_PROBE_N   = 10
PATCHINESS_LOW_FRAC  = 0.10
PATCHINESS_HIGH_FRAC = 0.90
PATCHINESS_VAR_THRESH = 0.05   # mag²

# HDF5 quality thresholds
MAX_SIGMA_MAG  = 0.3
MAX_FLAG_VALUE = 0

# Strategy colours / labels
STRATEGIES   = ("absolute", "motion", "scheduler")
STRAT_COLORS = {"absolute": "#2ca02c", "motion": "#1f77b4", "scheduler": "#d62728"}
STRAT_LABELS = {"absolute": "Abs Min (Green)",
                "motion":   "Motion Track (Blue)",
                "scheduler":"Scheduler (Red)"}

URL_COL  = "lsst.sal.DREAM.logevent_largeFileObjectAvailable.url"
TIME_COL = "time"


# ═══════════════════════════════════════════════════════════════════════════════
# UTILITY HELPERS
# ═══════════════════════════════════════════════════════════════════════════════

def _location():
    return EarthLocation(lat=RUBIN_LAT*u.deg, lon=RUBIN_LON*u.deg,
                         height=RUBIN_HEIGHT_M*u.m)


def _ensure_time(t):
    if not isinstance(t, Time):
        t = Time(t)
    return t.utc


def transform_url(url):
    url = str(url).strip()
    if url.startswith("https://s3.cp.lsst.org/"):
        return url.replace("https://s3.cp.lsst.org/", "s3://lfa@")
    return url


def fetch_cloud_map(url):
    """Load cloud_sys HDF5 → (clouds, sigma) arrays, bad pixels → NaN."""
    if HAS_LSST_RESOURCES:
        rp = ResourcePath(url)
        with rp.open("rb") as fd:
            data = fd.read()
    else:
        import urllib.request
        with urllib.request.urlopen(url) as resp:
            data = resp.read()

    with h5py.File(io.BytesIO(data), "r") as f:
        clouds = np.array(f["clouds"],       dtype=float).ravel()
        sigma  = np.array(f["sigma"],        dtype=float).ravel()
        flags  = np.array(f["flags"],        dtype=int  ).ravel()
        vis    = np.array(f["mask_visible"], dtype=bool ).ravel()
        nobs   = np.array(f["nobs"],         dtype=int  ).ravel()

    bad = ~vis | (nobs == 0) | (flags > MAX_FLAG_VALUE) | (sigma > MAX_SIGMA_MAG) | ~np.isfinite(clouds)
    clouds[bad] = np.nan
    sigma[bad]  = np.nan
    return clouds, sigma


def healpix_to_grid(clouds, obstime):
    """Project HEALPix map onto flat-sky km grid."""
    npix = hp.nside2npix(NSIDE)
    pix  = np.arange(npix)
    th, ph = hp.pix2ang(NSIDE, pix, nest=NEST)
    ra  = np.degrees(ph)
    dec = 90.0 - np.degrees(th)

    t   = _ensure_time(obstime)
    loc = _location()
    sky = SkyCoord(ra=ra*u.deg, dec=dec*u.deg, frame="icrs")
    aa  = sky.transform_to(AltAz(obstime=t, location=loc))
    alt = aa.alt.deg
    az  = aa.az.deg % 360.0

    alt_r  = np.radians(alt)
    az_r   = np.radians(az)
    above  = alt > MIN_ALT_DEG
    scale  = np.where(above, R_PROJECTION / np.sin(alt_r), np.nan)
    xf     = -np.cos(alt_r) * np.sin(az_r) * scale
    yf     =  np.cos(alt_r) * np.cos(az_r) * scale
    vals   = np.where(above, clouds, np.nan)

    ok = ~np.isnan(vals) & (np.sqrt(xf**2 + yf**2) <= 15_000)
    x_edges = np.arange(-15_000, 15_001, BIN_SIZE_KM)
    y_edges = np.arange(-15_000, 15_001, BIN_SIZE_KM)

    Hs, _, _ = np.histogram2d(xf[ok], yf[ok], bins=[x_edges, y_edges], weights=vals[ok])
    Hc, _, _ = np.histogram2d(xf[ok], yf[ok], bins=[x_edges, y_edges])
    with np.errstate(divide="ignore", invalid="ignore"):
        H = Hs / Hc
    H[Hc == 0] = np.nan
    H = H.T
    xc = (x_edges[:-1] + x_edges[1:]) / 2
    yc = (y_edges[:-1] + y_edges[1:]) / 2
    Xg, Yg = np.meshgrid(xc, yc)
    H[np.sqrt(Xg**2 + Yg**2) > 15_000] = np.nan
    return H


def xy_to_altaz(x_km, y_km):
    r   = np.sqrt(x_km**2 + y_km**2)
    alt = float(np.degrees(np.arctan2(R_PROJECTION, r)))
    az  = float(np.degrees(np.arctan2(-x_km, y_km)) % 360.0)
    return alt, az


def altaz_to_radec(alt_deg, az_deg, obstime):
    t   = _ensure_time(obstime)
    loc = _location()
    aa  = SkyCoord(alt=alt_deg*u.deg, az=az_deg*u.deg,
                   frame=AltAz(obstime=t, location=loc))
    icrs = aa.icrs
    return float(icrs.ra.deg), float(icrs.dec.deg)


def grid_value(grid, x_km, y_km):
    xi = int(round(x_km / BIN_SIZE_KM + 15))
    yi = int(round(y_km / BIN_SIZE_KM + 15))
    if 0 <= xi < grid.shape[1] and 0 <= yi < grid.shape[0]:
        return grid[yi, xi]
    return np.nan


def find_abs_min(grid):
    if not np.any(~np.isnan(grid)):
        return 0, 0, np.nan
    idx     = np.nanargmin(grid)
    yi, xi  = np.unravel_index(idx, grid.shape)
    return (xi - 15) * BIN_SIZE_KM, (yi - 15) * BIN_SIZE_KM, grid[yi, xi]


def radec_to_xy(ra_deg, dec_deg, obstime):
    t   = _ensure_time(obstime)
    loc = _location()
    sky = SkyCoord(ra=ra_deg*u.deg, dec=dec_deg*u.deg, frame="icrs")
    aa  = sky.transform_to(AltAz(obstime=t, location=loc))
    alt, az = float(aa.alt.deg), float(aa.az.deg % 360.0)
    if alt < MIN_ALT_DEG:
        return None, None, alt, az
    alt_r, az_r = np.radians(alt), np.radians(az)
    scale = R_PROJECTION / np.sin(alt_r)
    x = -np.cos(alt_r) * np.sin(az_r) * scale
    y =  np.cos(alt_r) * np.cos(az_r) * scale
    return float(x), float(y), alt, az


def slew_time(alt1, az1, alt2, az2):
    da  = abs(alt2 - alt1)
    daz = abs(az2 - az1)
    if daz > 180:
        daz = 360 - daz
    return max(da / MAX_SLEW_ALT, daz / MAX_SLEW_AZ) + SLEW_SETTLE


def extinction_to_m5(ext_mag, airmass=1.2, base_m5=24.5):
    return base_m5 - ext_mag - 0.1 * (airmass - 1.0)


def photon_count(ext_mag):
    if np.isnan(ext_mag):
        return np.nan
    flux = 10 ** (-0.4 * ext_mag)
    return PHOTON_FLUX_MAG20 * RUBIN_AREA * RUBIN_QE * flux * EXPOSURE_TIME


def cloud_motion(g1, g2, sigma=5.0, search=10):
    g1s = gaussian_filter(np.nan_to_num(g1, nan=0), sigma)
    g2s = gaussian_filter(np.nan_to_num(g2, nan=0), sigma)
    m1  = ~np.isnan(g1)
    m2  = ~np.isnan(g2)
    best, bdx, bdy = -np.inf, 0, 0
    for dy in range(-search, search + 1):
        for dx in range(-search, search + 1):
            sh = ndimage.shift(g2s, (dy, dx), order=1, mode="constant", cval=0)
            sm = ndimage.shift(m2.astype(float), (dy, dx), order=0, mode="constant", cval=0) > 0.5
            v  = m1 & sm
            if v.sum() < 100:
                continue
            v1, v2 = g1s[v], sh[v]
            if np.std(v1) > 0 and np.std(v2) > 0:
                c = np.corrcoef(v1, v2)[0, 1]
                if c > best:
                    best, bdx, bdy = c, dx, dy
    return bdx, bdy, best


# ═══════════════════════════════════════════════════════════════════════════════
# DATA LOADING
# ═══════════════════════════════════════════════════════════════════════════════

def load_all_sys_frames(csv_file):
    """Load and index entire CSV for cloud_sys frames, tagging each with a night_key."""
    print(f"\nIndexing DREAM CSV: {csv_file}")
    df = pd.read_csv(csv_file)
    df.columns = df.columns.str.replace('"', "").str.strip()
    df = df.dropna(subset=[URL_COL]).copy()
    df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors="coerce", utc=True)
    df = df.dropna(subset=[TIME_COL])
    df = df[df[URL_COL].str.contains(r"\.hdf5",   case=False, na=False, regex=True)]
    df = df[df[URL_COL].str.contains("cloud_sys", case=False, na=False)]
    df = df.sort_values(TIME_COL).reset_index(drop=True)
    # Frames before noon UTC belong to the previous calendar night
    shifted = df[TIME_COL] - pd.Timedelta(hours=12)
    df["night_key"] = shifted.dt.date
    all_nights = sorted(df["night_key"].unique())
    print(f"  {len(df)} cloud_sys frames across {len(all_nights)} nights:")
    for nk in all_nights:
        n = (df["night_key"] == nk).sum()
        print(f"    {nk}  ({n} frames)")
    return df


def load_scheduler_db(db_file):
    """Load one full night of scheduler pointings from an opsim SQLite file."""
    print(f"\nLoading scheduler DB: {db_file}")
    conn   = sqlite3.connect(db_file)
    cursor = conn.cursor()
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = [r[0] for r in cursor.fetchall()]
    table  = next((n for n in ["observations", "SummaryAllProps", "Summary", "obs"]
                   if n in tables), tables[0])
    obs    = pd.read_sql_query(f"SELECT * FROM {table}", conn)
    conn.close()

    night_col = next((c for c in obs.columns if "night" in c.lower()), None)
    obs["night"] = obs[night_col].astype(int)
    for want, candidates in {
        "observationStartMJD": ["observationStartMJD", "observationstartmjd", "mjd"],
        "fieldRA":  ["fieldRA",  "fieldra",  "ra"],
        "fieldDec": ["fieldDec", "fielddec", "dec"],
        "filter":   ["filter",   "band"],
        "airmass":  ["airmass"],
        "seeingFwhmEff":    ["seeingFwhmEff", "seeingfwhmeff", "seeing"],
        "fiveSigmaDepth":   ["fiveSigmaDepth", "fivesigmadepth", "m5"],
        "slewTime":         ["slewTime", "slewtime"],
        "visitExposureTime":["visitExposureTime", "visitexposuretime", "exptime"],
    }.items():
        if want not in obs.columns:
            for c in candidates:
                if c in obs.columns:
                    obs[want] = obs[c]
                    break

    scores = {}
    for n in obs["night"].unique()[:500]:
        sub = obs[obs["night"] == n]
        if len(sub) < 200:
            continue
        dur = (sub["observationStartMJD"].max() - sub["observationStartMJD"].min()) * 24
        if dur < 4:
            continue
        scores[n] = len(sub) * 10 + dur

    chosen = max(scores, key=scores.get) if scores else obs.groupby("night").size().idxmax()
    night  = obs[obs["night"] == chosen].sort_values("observationStartMJD").reset_index(drop=True)
    dur    = (night["observationStartMJD"].max() - night["observationStartMJD"].min()) * 24
    print(f"  Scheduler night {chosen}: {len(night)} obs, {dur:.1f} h")
    return night


def match_sched_to_frames(sched, frame_times):
    n        = len(frame_times)
    f_fracs  = np.linspace(0, 1, n)
    mjd      = sched["observationStartMJD"].values
    rng      = mjd.max() - mjd.min() + 1e-12
    s_fracs  = (mjd - mjd.min()) / rng
    return [sched.iloc[np.argmin(np.abs(s_fracs - f))] for f in f_fracs]


# ═══════════════════════════════════════════════════════════════════════════════
# PATCHINESS CHECK
# ═══════════════════════════════════════════════════════════════════════════════

def patchiness_probe(df_night, n_probe=PATCHINESS_PROBE_N):
    """
    Fetch ~n_probe evenly-spaced frames and compute spatial variance.
    Returns (patchy: bool, mean_frac, mean_var, reason, probe_grids).
    """
    if len(df_night) == 0:
        return False, 0.0, 0.0, "No frames", []

    idx_arr = np.linspace(0, len(df_night) - 1,
                          min(n_probe, len(df_night)), dtype=int)
    grids = []
    for i in idx_arr:
        row = df_night.iloc[i]
        url = transform_url(row[URL_COL])
        try:
            clouds, _ = fetch_cloud_map(url)
            g = healpix_to_grid(clouds, row[TIME_COL].to_pydatetime())
            grids.append(g)
        except Exception as e:
            print(f"    probe frame {i} failed: {e}")

    if len(grids) < 2:
        return False, 0.0, 0.0, "Too few probe frames", []

    fracs, vars_ = [], []
    for g in grids:
        valid = ~np.isnan(g)
        fracs.append(np.sum(valid) / g.size)
        vars_.append(float(np.nanvar(g)) if valid.sum() > 10 else 0.0)

    mf = float(np.mean(fracs))
    mv = float(np.mean(vars_))

    if mf < PATCHINESS_LOW_FRAC:
        return False, mf, mv, f"Mostly clear (frac={mf:.2f})", grids
    if mf > PATCHINESS_HIGH_FRAC:
        return False, mf, mv, f"Overcast (frac={mf:.2f})", grids
    if mv < PATCHINESS_VAR_THRESH:
        return False, mf, mv, f"Uniform cover (var={mv:.4f})", grids
    return True, mf, mv, f"Patchy (frac={mf:.2f}, var={mv:.4f})", grids


# ═══════════════════════════════════════════════════════════════════════════════
# BUILD VISIT TABLES
# ═══════════════════════════════════════════════════════════════════════════════

def build_visit_tables(df_night, sched_obs, max_frames=None):
    """
    For one night, build three visit DataFrames (one per strategy)
    mimicking an opsim output table.
    """
    if max_frames:
        df_night = df_night.iloc[:max_frames]

    matched  = match_sched_to_frames(sched_obs, df_night[TIME_COL].tolist())
    rows     = {s: [] for s in STRATEGIES}
    prev_alt = {s: 90.0 for s in STRATEGIES}
    prev_az  = {s:  0.0 for s in STRATEGIES}

    prev_grid    = None
    xp, yp       = 0.0, 0.0
    dx_hist, dy_hist = [], []
    HL = 3

    print(f"    Building visit tables for {len(df_night)} frames …")
    for idx, (_, row) in enumerate(df_night.iterrows()):
        if idx % 50 == 0:
            print(f"      Frame {idx}/{len(df_night)}")

        url     = transform_url(row[URL_COL])
        t_visit = row[TIME_COL]

        try:
            clouds, _ = fetch_cloud_map(url)
        except Exception as e:
            print(f"      skip frame {idx}: {e}")
            prev_grid = None
            continue

        grid      = healpix_to_grid(clouds, t_visit)
        t_astropy = _ensure_time(t_visit)
        mjd       = float(t_astropy.mjd)

        # ── Absolute minimum ──────────────────────────────────────────
        xa, ya, ext_a = find_abs_min(grid)
        alt_a, az_a   = xy_to_altaz(xa, ya)
        if not np.isnan(ext_a) and alt_a > MIN_ALT_DEG:
            slew_a = slew_time(prev_alt["absolute"], prev_az["absolute"], alt_a, az_a)
            ra_a, dec_a = altaz_to_radec(alt_a, az_a, t_visit)
            am_a = 1.0 / np.sin(np.radians(alt_a))
            rows["absolute"].append(dict(
                observationStartMJD=mjd, fieldRA=ra_a, fieldDec=dec_a,
                filter="r", airmass=am_a,
                fiveSigmaDepth=extinction_to_m5(ext_a, am_a),
                seeingFwhmEff=0.7 + 0.1*(am_a - 1),
                slewTime=slew_a, visitExposureTime=EXPOSURE_TIME,
                clouds_ext=ext_a, photons=photon_count(ext_a), night=0,
            ))
            prev_alt["absolute"], prev_az["absolute"] = alt_a, az_a

        # ── Motion tracking ───────────────────────────────────────────
        if prev_grid is not None:
            dx, dy, conf = cloud_motion(prev_grid, grid)
            if conf > 0.5:
                dx_hist.append(dx); dy_hist.append(dy)
            if len(dx_hist) > HL:
                dx_hist.pop(0); dy_hist.pop(0)
            dx_avg = float(np.mean(dx_hist)) if dx_hist else 0.0
            dy_avg = float(np.mean(dy_hist)) if dy_hist else 0.0
            xp_new = xp - dx_avg * BIN_SIZE_KM
            yp_new = yp - dy_avg * BIN_SIZE_KM
            r_new  = np.sqrt(xp_new**2 + yp_new**2)
            if r_new > 14_000:
                xp_new *= 14_000 / r_new; yp_new *= 14_000 / r_new
            val_new = grid_value(grid, xp_new, yp_new)
            if np.isnan(val_new) or val_new > 0.5:
                xp_new, yp_new, _ = find_abs_min(grid)
            xp, yp = xp_new, yp_new
        else:
            xp, yp, _ = find_abs_min(grid)

        ext_p = grid_value(grid, xp, yp)
        alt_p, az_p = xy_to_altaz(xp, yp)
        if not np.isnan(ext_p) and alt_p > MIN_ALT_DEG:
            slew_p = slew_time(prev_alt["motion"], prev_az["motion"], alt_p, az_p)
            ra_p, dec_p = altaz_to_radec(alt_p, az_p, t_visit)
            am_p = 1.0 / np.sin(np.radians(alt_p))
            rows["motion"].append(dict(
                observationStartMJD=mjd, fieldRA=ra_p, fieldDec=dec_p,
                filter="r", airmass=am_p,
                fiveSigmaDepth=extinction_to_m5(ext_p, am_p),
                seeingFwhmEff=0.7 + 0.1*(am_p - 1),
                slewTime=slew_p, visitExposureTime=EXPOSURE_TIME,
                clouds_ext=ext_p, photons=photon_count(ext_p), night=0,
            ))
            prev_alt["motion"], prev_az["motion"] = alt_p, az_p

        prev_grid = grid.copy()

        # ── Scheduler ─────────────────────────────────────────────────
        obs = matched[idx]
        xs, ys, alt_s, az_s = radec_to_xy(
            float(obs["fieldRA"]), float(obs["fieldDec"]), t_visit)
        if xs is not None:
            ext_s = grid_value(grid, xs, ys)
            if not np.isnan(ext_s) and alt_s > MIN_ALT_DEG:
                slew_s = slew_time(prev_alt["scheduler"], prev_az["scheduler"], alt_s, az_s)
                am_s   = 1.0 / np.sin(np.radians(alt_s))
                raw_m5 = obs.get("fiveSigmaDepth", np.nan)
                if pd.isna(raw_m5):
                    raw_m5 = extinction_to_m5(ext_s, am_s)
                rows["scheduler"].append(dict(
                    observationStartMJD=mjd,
                    fieldRA=float(obs["fieldRA"]), fieldDec=float(obs["fieldDec"]),
                    filter=str(obs.get("filter", "r")).strip("'"),
                    airmass=am_s,
                    fiveSigmaDepth=float(raw_m5) - ext_s,
                    seeingFwhmEff=float(obs.get("seeingFwhmEff", 0.8) or 0.8),
                    slewTime=slew_s, visitExposureTime=EXPOSURE_TIME,
                    clouds_ext=ext_s, photons=photon_count(ext_s), night=0,
                ))
                prev_alt["scheduler"], prev_az["scheduler"] = alt_s, az_s

    dfs = {s: pd.DataFrame(rows[s]) for s in STRATEGIES}
    for s, df in dfs.items():
        print(f"      {s:10s}: {len(df):4d} visits")
    return dfs


# ═══════════════════════════════════════════════════════════════════════════════
# LIGHTWEIGHT METRICS
# ═══════════════════════════════════════════════════════════════════════════════

def lightweight_metrics(dfs):
    """Compute per-strategy scalar metrics without rubin_sim.maf."""
    results = {}

    for s, df in dfs.items():
        if df.empty:
            continue
        mjd  = df["observationStartMJD"].values
        sort_idx = np.argsort(mjd)
        mjd  = mjd[sort_idx]
        ph   = df["photons"].values[sort_idx]
        m5   = df["fiveSigmaDepth"].values[sort_idx]
        am   = df["airmass"].values[sort_idx]
        see  = df["seeingFwhmEff"].values[sort_idx]
        slew = df["slewTime"].values[sort_idx]
        ext  = df["clouds_ext"].values[sort_idx]

        results.setdefault("N_visits",          {})[s] = len(df)
        results.setdefault("Total_photons",     {})[s] = float(np.nansum(ph))

        dur_h = (mjd[-1] - mjd[0]) * 24 if len(mjd) > 1 else 0.0
        total_slew_h = float(np.sum(slew)) / 3600.0
        exp_h = len(df) * EXPOSURE_TIME / 3600.0
        results.setdefault("Night_duration_h",  {})[s] = dur_h
        results.setdefault("Total_slew_h",      {})[s] = total_slew_h
        results.setdefault("Total_exposure_h",  {})[s] = exp_h
        results.setdefault("Open_shutter_frac", {})[s] = float(exp_h / (dur_h + 1e-12))

        results.setdefault("Median_airmass",    {})[s] = float(np.nanmedian(am))
        results.setdefault("Max_airmass",       {})[s] = float(np.nanmax(am))
        results.setdefault("Mean_airmass",      {})[s] = float(np.nanmean(am))

        results.setdefault("Median_seeing",     {})[s] = float(np.nanmedian(see))

        results.setdefault("Median_m5",         {})[s] = float(np.nanmedian(m5))
        results.setdefault("Mean_m5",           {})[s] = float(np.nanmean(m5))
        results.setdefault("Coadd_m5_proxy",    {})[s] = float(
            -2.5 * np.log10(np.nansum(10 ** (-0.4 * m5 * 2))) / 2
            if len(m5) else np.nan)

        results.setdefault("Mean_cloud_ext",    {})[s] = float(np.nanmean(ext))
        results.setdefault("Median_cloud_ext",  {})[s] = float(np.nanmedian(ext))
        results.setdefault("Min_cloud_ext",     {})[s] = float(np.nanmin(ext))

        if len(mjd) > 1:
            gaps = np.diff(mjd) * 24 * 60
            results.setdefault("Max_gap_min",    {})[s] = float(gaps.max())
            results.setdefault("Median_gap_min", {})[s] = float(np.median(gaps))
        else:
            results.setdefault("Max_gap_min",    {})[s] = np.nan
            results.setdefault("Median_gap_min", {})[s] = np.nan

        results.setdefault("Mean_slew_s",       {})[s] = float(np.nanmean(slew))
        results.setdefault("Median_slew_s",     {})[s] = float(np.nanmedian(slew))

        pix_set = set()
        for _, r in df.iterrows():
            th  = np.radians(90.0 - r["fieldDec"])
            phi = np.radians(r["fieldRA"] % 360.0)
            pix_set.add(hp.ang2pix(NSIDE, th, phi, nest=NEST))
        results.setdefault("Sky_frac_visited",  {})[s] = float(len(pix_set) / hp.nside2npix(NSIDE))
        results.setdefault("fO_area_deg2",      {})[s] = float(
            len(pix_set) * hp.nside2pixarea(NSIDE, degrees=True))

        phot_m5 = extinction_to_m5(0.0)
        results.setdefault("Delta_depth_vs_phot",{})[s] = float(np.nanmean(m5) - phot_m5)

    return results


# ═══════════════════════════════════════════════════════════════════════════════
# FULL MAF METRICS
# ═══════════════════════════════════════════════════════════════════════════════

def run_maf_metrics(dfs, out_dir="maf_outputs"):
    """Run native rubin_sim.maf metrics; returns dict of scalars per strategy."""
    import tempfile
    os.makedirs(out_dir, exist_ok=True)

    REQUIRED = [
        "observationId", "fieldRA", "fieldDec", "filter",
        "observationStartMJD", "visitExposureTime",
        "airmass", "seeingFwhmEff", "fiveSigmaDepth",
        "slewTime", "night",
        "rotSkyPos", "numExposures", "skyBrightness",
        "slewDistance", "altitude", "azimuth",
        "moonAlt", "sunAlt", "cloud",
    ]

    maf_results = {}
    for s, df in dfs.items():
        if df.empty:
            maf_results[s] = {}
            continue

        print(f"    MAF on {s} ({len(df)} visits) …")
        full = df.copy()
        full["observationId"]   = np.arange(len(full), dtype=int)
        full["rotSkyPos"]       = 0.0
        full["numExposures"]    = 1
        full["skyBrightness"]   = 21.0
        full["slewDistance"]    = full["slewTime"] * 3.5
        full["altitude"]        = np.degrees(np.arcsin(1.0 / full["airmass"].clip(1, 5)))
        full["azimuth"]         = 180.0
        full["moonAlt"]         = -30.0
        full["sunAlt"]          = -18.0
        full["cloud"]           = full["clouds_ext"].fillna(0.5)
        for col in REQUIRED:
            if col not in full.columns:
                full[col] = 0.0

        with tempfile.NamedTemporaryFile(suffix=".db", delete=False) as tmp:
            db_path = tmp.name
        conn = sqlite3.connect(db_path)
        full[REQUIRED].to_sql("observations", conn, if_exists="replace", index=False)
        conn.close()

        strat_out = os.path.join(out_dir, s)
        os.makedirs(strat_out, exist_ok=True)
        uni     = maf.UniSlicer()
        healpix = maf.HealpixSlicer(nside=NSIDE)
        bundles = [
            maf.MetricBundle(maf.CountMetric("observationId"),       uni,     "", run_name=s),
            maf.MetricBundle(maf.MedianMetric("airmass"),             uni,     "", run_name=s),
            maf.MetricBundle(maf.MaxMetric("airmass"),                uni,     "", run_name=s),
            maf.MetricBundle(maf.MedianMetric("seeingFwhmEff"),       uni,     "", run_name=s),
            maf.MetricBundle(maf.MedianMetric("fiveSigmaDepth"),      uni,     "", run_name=s),
            maf.MetricBundle(maf.MeanMetric("fiveSigmaDepth"),        uni,     "", run_name=s),
            maf.MetricBundle(maf.MeanMetric("slewTime"),              uni,     "", run_name=s),
            maf.MetricBundle(maf.OpenShutterFractionMetric(),         uni,     "", run_name=s),
            maf.MetricBundle(maf.MaxGapMetric(),                      uni,     "", run_name=s),
            maf.MetricBundle(maf.Coaddm5Metric(),                 healpix,    "", run_name=s),
            maf.MetricBundle(
                maf.fOArea(col="observationId", nside=NSIDE),     healpix,    "", run_name=s),
        ]
        bg = maf.MetricBundleGroup(bundles, db_path, out_dir=strat_out)
        bg.run_all()
        os.unlink(db_path)

        scalars = {}
        for b in bundles:
            mv = b.metric_values
            if hasattr(mv, "compressed") and mv.compressed().size == 1:
                scalars[b.metric.name] = float(mv.compressed()[0])
            elif hasattr(mv, "data") and mv.data.size == 1:
                scalars[b.metric.name] = float(mv.data.ravel()[0])
        maf_results[s] = scalars

    return maf_results


# ═══════════════════════════════════════════════════════════════════════════════
# PER-NIGHT PLOTS
# ═══════════════════════════════════════════════════════════════════════════════

def plot_night_summary(results, dfs, night_label, output):
    fig = plt.figure(figsize=(22, 18))
    fig.suptitle(
        f"MAF-Style Metric Comparison — Night {night_label}\n"
        "DREAM Cloud Scheduling Strategies",
        fontsize=15, weight="bold", y=0.98)

    gs = gridspec.GridSpec(4, 3, figure=fig, hspace=0.45, wspace=0.35)
    colors = [STRAT_COLORS[s] for s in STRATEGIES]
    labels = [STRAT_LABELS[s]  for s in STRATEGIES]

    def bar_panel(ax, key, ylabel, title, higher_better=True, scale=1.0):
        vals = [results.get(key, {}).get(s, np.nan) * scale for s in STRATEGIES]
        finite = [v for v in vals if not np.isnan(v)]
        if not finite:
            ax.set_title(title + " (no data)"); return
        bars = ax.bar(labels, vals, color=colors, alpha=0.78,
                      edgecolor="black", linewidth=1.2)
        best = max(finite) if higher_better else min(finite)
        for bar, val in zip(bars, vals):
            if np.isnan(val):
                continue
            ax.text(bar.get_x() + bar.get_width()/2, val,
                    f"{val:.3f}", ha="center", va="bottom", fontsize=8.5, weight="bold")
            if abs(val - best) < 1e-9:
                bar.set_edgecolor("gold"); bar.set_linewidth(3)
        ax.set_ylabel(ylabel, fontsize=9)
        ax.set_title(title, fontsize=10, weight="bold")
        ax.grid(axis="y", alpha=0.3)
        ax.tick_params(axis="x", labelsize=7.5, rotation=15)

    bar_panel(fig.add_subplot(gs[0, 0]), "N_visits",          "Count",    "N Visits",           True)
    bar_panel(fig.add_subplot(gs[0, 1]), "Total_photons",     "Photons",  "Total Photons (×10⁶)",True, 1e-6)
    bar_panel(fig.add_subplot(gs[0, 2]), "Open_shutter_frac", "Fraction", "Open-Shutter Frac",  True)
    bar_panel(fig.add_subplot(gs[1, 0]), "Median_m5",         "mag",      "Median 5σ Depth",    True)
    bar_panel(fig.add_subplot(gs[1, 1]), "Median_airmass",    "Airmass",  "Median Airmass",     False)
    bar_panel(fig.add_subplot(gs[1, 2]), "Mean_cloud_ext",    "mag",      "Mean Cloud Ext",     False)
    bar_panel(fig.add_subplot(gs[2, 0]), "Mean_slew_s",       "s",        "Mean Slew Time",     False)
    bar_panel(fig.add_subplot(gs[2, 1]), "Total_slew_h",      "h",        "Total Slew Time",    False)
    bar_panel(fig.add_subplot(gs[2, 2]), "fO_area_deg2",      "deg²",     "Sky Area Visited",   True)

    ax_ts  = fig.add_subplot(gs[3, :2])
    ax_m5h = fig.add_subplot(gs[3, 2])
    for s in STRATEGIES:
        df = dfs[s]
        if df.empty:
            continue
        mjd = df["observationStartMJD"].values
        ph  = df["photons"].values
        t_h = (mjd - mjd.min()) * 24
        ax_ts.plot(t_h, ph, color=STRAT_COLORS[s], alpha=0.6, lw=1.2, label=STRAT_LABELS[s])
        m5  = df["fiveSigmaDepth"].dropna()
        ax_m5h.hist(m5, bins=25, color=STRAT_COLORS[s], alpha=0.5,
                    label=STRAT_LABELS[s], histtype="stepfilled", edgecolor="black", lw=0.8)

    ax_ts.set_xlabel("Hours into night", fontsize=9)
    ax_ts.set_ylabel("Photons / 30 s", fontsize=9)
    ax_ts.set_title("Photon Count Per Exposure", fontsize=10, weight="bold")
    ax_ts.legend(fontsize=7); ax_ts.grid(alpha=0.3)
    ax_ts.ticklabel_format(axis="y", style="scientific", scilimits=(0,0))
    ax_m5h.set_xlabel("5σ Limiting Magnitude", fontsize=9)
    ax_m5h.set_ylabel("Count", fontsize=9)
    ax_m5h.set_title("Survey Depth Distribution", fontsize=10, weight="bold")
    ax_m5h.legend(fontsize=7); ax_m5h.grid(alpha=0.3)

    plt.savefig(output, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"    → {output}")


def plot_sky_maps(dfs, night_label, output):
    fig, axes = plt.subplots(1, 3, figsize=(21, 5),
                             subplot_kw={"projection": "mollweide"})
    fig.suptitle(f"Coadded 5σ Depth Sky Maps — Night {night_label}",
                 fontsize=13, weight="bold", y=1.01)
    vmin, vmax = 22.0, 26.0
    for ax, s in zip(axes, STRATEGIES):
        df = dfs[s]
        ras, decs, m5s = [], [], []
        if not df.empty:
            for _, r in df.iterrows():
                if np.isfinite(r["fiveSigmaDepth"]):
                    ras.append(r["fieldRA"])
                    decs.append(r["fieldDec"])
                    m5s.append(r["fiveSigmaDepth"])

        ax.set_title(STRAT_LABELS[s], fontsize=10, weight="bold", pad=6)
        ax.grid(True, alpha=0.3, lw=0.5)
        if ras:
            ra_rad  = ((np.radians(np.array(ras)) + np.pi) % (2*np.pi)) - np.pi
            dec_rad = np.radians(np.array(decs))
            sc = ax.scatter(ra_rad, dec_rad, c=np.array(m5s),
                            cmap="viridis", vmin=vmin, vmax=vmax, s=6, alpha=0.8, lw=0)
        else:
            sc = ax.scatter([], [], c=[], cmap="viridis", vmin=vmin, vmax=vmax, s=6)
            ax.text(0, 0, "No data", ha="center", va="center",
                    fontsize=10, color="gray")
        plt.colorbar(sc, ax=ax, orientation="horizontal", pad=0.06,
                     fraction=0.046, label="coadd 5σ (mag)")

    plt.tight_layout()
    plt.savefig(output, dpi=130, bbox_inches="tight")
    plt.close()
    print(f"    → {output}")


# ═══════════════════════════════════════════════════════════════════════════════
# CROSS-NIGHT SUMMARY PLOT
# ═══════════════════════════════════════════════════════════════════════════════

def plot_cross_night_summary(all_night_results, output):
    """
    Multi-panel chart summarising each strategy's performance across all
    patchy nights.  all_night_results: list of dicts with keys
    'night', 'results' (metric dict), 'dfs'.
    """
    if not all_night_results:
        print("No patchy nights to summarise.")
        return

    night_labels = [r["night"] for r in all_night_results]
    x = np.arange(len(night_labels))
    width = 0.25

    # Key metrics to panel
    PANELS = [
        ("N_visits",          "N Visits",              True,  1.0),
        ("Total_photons",     "Total Photons (×10⁶)",  True,  1e-6),
        ("Open_shutter_frac", "Open-Shutter Fraction", True,  1.0),
        ("Median_m5",         "Median 5σ Depth (mag)", True,  1.0),
        ("Mean_cloud_ext",    "Mean Cloud Ext (mag)",  False, 1.0),
        ("fO_area_deg2",      "Sky Area (deg²)",       True,  1.0),
    ]

    fig, axes = plt.subplots(2, 3, figsize=(22, 12))
    fig.suptitle(
        "Cross-Night MAF Strategy Comparison\n"
        "All Partially-Cloudy Nights  |  DREAM Cloud Scheduling",
        fontsize=14, weight="bold")
    axes = axes.ravel()

    for ax, (key, ylabel, higher_better, scale) in zip(axes, PANELS):
        for offset, s in zip([-width, 0, width], STRATEGIES):
            vals = [r["results"].get(key, {}).get(s, np.nan) * scale
                    for r in all_night_results]
            bars = ax.bar(x + offset, vals, width,
                          label=STRAT_LABELS[s],
                          color=STRAT_COLORS[s], alpha=0.75,
                          edgecolor="black", linewidth=0.8)
        ax.set_xticks(x)
        ax.set_xticklabels(night_labels, rotation=30, ha="right", fontsize=8)
        ax.set_ylabel(ylabel, fontsize=9)
        ax.set_title(ylabel, fontsize=10, weight="bold")
        ax.legend(fontsize=7, loc="best")
        ax.grid(axis="y", alpha=0.3)

    plt.tight_layout()
    plt.savefig(output, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"\n  → Cross-night summary plot: {output}")


def plot_cross_night_timeseries(all_night_results, output):
    """
    Line chart of cumulative photons and mean depth over all patchy nights.
    """
    if not all_night_results:
        return

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))
    fig.suptitle("Cumulative Performance Across Patchy Nights", fontsize=13, weight="bold")

    night_labels = [r["night"] for r in all_night_results]
    x = np.arange(len(night_labels))

    for s in STRATEGIES:
        cum_ph  = np.cumsum([
            r["results"].get("Total_photons", {}).get(s, 0) or 0
            for r in all_night_results])
        med_m5  = [r["results"].get("Median_m5", {}).get(s, np.nan)
                   for r in all_night_results]

        ax1.plot(x, cum_ph * 1e-9, "o-", color=STRAT_COLORS[s],
                 label=STRAT_LABELS[s], lw=2, ms=7)
        ax2.plot(x, med_m5, "o-", color=STRAT_COLORS[s],
                 label=STRAT_LABELS[s], lw=2, ms=7)

    ax1.set_xticks(x); ax1.set_xticklabels(night_labels, rotation=30, ha="right", fontsize=8)
    ax1.set_ylabel("Cumulative photons (×10⁹)", fontsize=10)
    ax1.set_title("Cumulative Photon Collection", fontsize=11, weight="bold")
    ax1.legend(fontsize=9); ax1.grid(alpha=0.3)

    ax2.set_xticks(x); ax2.set_xticklabels(night_labels, rotation=30, ha="right", fontsize=8)
    ax2.set_ylabel("Median 5σ Depth (mag)", fontsize=10)
    ax2.set_title("Per-Night Median Survey Depth", fontsize=11, weight="bold")
    ax2.legend(fontsize=9); ax2.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(output, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  → Cross-night timeseries: {output}")


# ═══════════════════════════════════════════════════════════════════════════════
# PER-NIGHT PIPELINE
# ═══════════════════════════════════════════════════════════════════════════════

def run_one_night(night_key, df_night, sched_obs, out_dir, max_frames=None):
    """
    Full MAF pipeline for one night.
    Returns (results_dict, dfs) or (None, None) if night is skipped.
    """
    tag        = str(night_key).replace("-", "")
    night_dir  = os.path.join(out_dir, tag)
    os.makedirs(night_dir, exist_ok=True)

    print(f"\n{'█'*70}")
    print(f"  NIGHT: {night_key}  ({len(df_night)} frames)")
    print(f"{'█'*70}")

    if len(df_night) < 5:
        print("  Too few frames — skipping.")
        return None, None

    # Patchiness probe
    print(f"  [Patchiness probe — {PATCHINESS_PROBE_N} frames]")
    patchy, mf, mv, reason, _ = patchiness_probe(df_night)
    print(f"    valid_frac={mf:.3f}  spatial_var={mv:.4f} mag²  → {reason}")
    if not patchy:
        print("  Skipping night (not patchy).")
        return None, None

    # Build visit tables
    print("  [Building visit tables]")
    dfs = build_visit_tables(df_night, sched_obs, max_frames=max_frames)

    # Check all three strategies produced data
    any_empty = [s for s in STRATEGIES if dfs[s].empty]
    if len(any_empty) == len(STRATEGIES):
        print("  All strategies empty — skipping.")
        return None, None
    if any_empty:
        print(f"  WARNING: {any_empty} produced no visits — included in summary with NaN.")

    # Lightweight metrics (always)
    print("  [Lightweight metrics]")
    results = lightweight_metrics(dfs)

    # Native MAF metrics (if available)
    if HAS_MAF:
        print("  [Native MAF metrics]")
        try:
            maf_r = run_maf_metrics(dfs, out_dir=os.path.join(night_dir, "maf_outputs"))
            results.update(maf_r)
        except Exception as e:
            print(f"  MAF run failed: {e}")

    # Save per-night CSV
    csv_path = os.path.join(night_dir, f"maf_results_{tag}.csv")
    rows = [{"metric": k, **v} for k, v in sorted(results.items())]
    pd.DataFrame(rows).to_csv(csv_path, index=False)
    print(f"  → {csv_path}")

    # Per-night plots
    plot_night_summary(results, dfs, str(night_key),
                       os.path.join(night_dir, f"metrics_{tag}.png"))
    try:
        plot_sky_maps(dfs, str(night_key),
                      os.path.join(night_dir, f"sky_maps_{tag}.png"))
    except Exception as e:
        print(f"  Sky map failed: {e}")

    return results, dfs


# ═══════════════════════════════════════════════════════════════════════════════
# CROSS-NIGHT SUMMARY TABLE
# ═══════════════════════════════════════════════════════════════════════════════

def build_cross_night_csv(all_night_results, output_csv):
    """Flatten all-night results into a tidy CSV (one row per night×strategy)."""
    rows = []
    key_metrics = [
        "N_visits", "Total_photons", "Open_shutter_frac",
        "Median_m5", "Mean_m5", "Coadd_m5_proxy",
        "Median_airmass", "Mean_cloud_ext", "Min_cloud_ext",
        "Mean_slew_s", "Total_slew_h", "Total_exposure_h",
        "Sky_frac_visited", "fO_area_deg2", "Delta_depth_vs_phot",
        "Max_gap_min", "Median_gap_min",
    ]
    for entry in all_night_results:
        night   = entry["night"]
        results = entry["results"]
        for s in STRATEGIES:
            row = {"night": night, "strategy": s, "strategy_label": STRAT_LABELS[s]}
            for k in key_metrics:
                row[k] = results.get(k, {}).get(s, np.nan)
            rows.append(row)

    df = pd.DataFrame(rows)
    df.to_csv(output_csv, index=False)
    print(f"\n  Cross-night CSV → {output_csv}")
    return df


def print_cross_night_table(df_summary):
    """Pretty-print aggregate statistics across all patchy nights."""
    print("\n" + "═"*80)
    print("  CROSS-NIGHT AGGREGATE SUMMARY")
    print("═"*80)
    agg = (df_summary.groupby("strategy")
           [["N_visits","Total_photons","Open_shutter_frac",
             "Median_m5","Mean_cloud_ext","fO_area_deg2"]]
           .agg(["mean","std"]))
    print(agg.to_string())
    print("\n  Wins per strategy across all patchy nights:")
    wins_total = {s: 0 for s in STRATEGIES}
    metric_cols = [c for c in df_summary.columns
                   if c not in ("night","strategy","strategy_label")]
    night_groups = df_summary.groupby("night")
    for _, gdf in night_groups:
        for col in metric_cols:
            vals = {row["strategy"]: row[col]
                    for _, row in gdf.iterrows()
                    if not pd.isna(row[col])}
            if not vals:
                continue
            higher = any(k in col for k in
                         ("photon","N_vis","m5","exposure","frac","sky","fO","depth","open"))
            winner = max(vals, key=vals.get) if higher else min(vals, key=vals.get)
            wins_total[winner] += 1
    for s in STRATEGIES:
        print(f"    {STRAT_LABELS[s]:<30s}: {wins_total[s]:3d} metric-night wins")
    print("═"*80)


# ═══════════════════════════════════════════════════════════════════════════════
# MAIN
# ═══════════════════════════════════════════════════════════════════════════════

def main(
    csv        = DEFAULT_CSV,
    db         = DEFAULT_DB,
    max_frames = None,
    out_dir    = DEFAULT_OUTDIR,
    _args      = None,
):
    """
    Entry point — callable from Python/Jupyter or the command line.

    Jupyter usage:
        from maf_multinight_scheduling import main
        main(csv="feb5_data.csv", db="baseline_v5.1.0_10yrs.db", max_frames=200)
    """
    if _args is not None:
        csv        = _args.csv
        db         = _args.db
        max_frames = _args.max_frames
        out_dir    = _args.out_dir

    os.makedirs(out_dir, exist_ok=True)

    print("\n" + "═"*70)
    print("  MAF METRICS — ALL PARTIALLY-CLOUDY NIGHTS")
    print(f"  CSV: {csv}   DB: {db}")
    print("═"*70)

    # ── 1. Index all cloud_sys frames ────────────────────────────────────────
    all_sys   = load_all_sys_frames(csv)
    all_nights = sorted(all_sys["night_key"].unique())

    # ── 2. Load scheduler once ───────────────────────────────────────────────
    sched_obs = load_scheduler_db(db)

    # ── 3. Iterate over nights ───────────────────────────────────────────────
    all_night_results = []   # list of {"night", "results", "dfs"}
    skipped = []

    for night_key in all_nights:
        df_night = all_sys[all_sys["night_key"] == night_key].reset_index(drop=True)
        results, dfs = run_one_night(
            night_key, df_night, sched_obs, out_dir, max_frames=max_frames)

        if results is not None:
            all_night_results.append({
                "night":   str(night_key),
                "results": results,
                "dfs":     dfs,
            })
        else:
            skipped.append(str(night_key))

    # ── 4. Cross-night outputs ───────────────────────────────────────────────
    print(f"\n{'═'*70}")
    print(f"  PROCESSED {len(all_night_results)} patchy nights  |  "
          f"skipped {len(skipped)} nights")
    if skipped:
        print("  Skipped:", ", ".join(skipped))
    print("═"*70)

    if not all_night_results:
        print("No patchy nights found. Done.")
        return

    df_summary = build_cross_night_csv(
        all_night_results,
        os.path.join(out_dir, "all_nights_summary.csv"))

    print_cross_night_table(df_summary)

    plot_cross_night_summary(
        all_night_results,
        os.path.join(out_dir, "cross_night_comparison.png"))

    plot_cross_night_timeseries(
        all_night_results,
        os.path.join(out_dir, "cross_night_timeseries.png"))

    print("\nDone.  All outputs under:", os.path.abspath(out_dir))


def _cli_main():
    import sys
    parser = argparse.ArgumentParser(
        description="MAF metrics on DREAM cloud scheduling – all patchy nights")
    parser.add_argument("--csv",        default=DEFAULT_CSV)
    parser.add_argument("--db",         default=DEFAULT_DB)
    parser.add_argument("--max-frames", type=int, default=None, dest="max_frames")
    parser.add_argument("--out-dir",    default=DEFAULT_OUTDIR, dest="out_dir")

    known_flags = {"--csv", "--db", "--max-frames", "--out-dir", "-h", "--help"}
    clean, skip = [], False
    for tok in sys.argv[1:]:
        if skip:
            skip = False; continue
        if tok in known_flags or any(tok.startswith(f + "=") for f in known_flags):
            clean.append(tok)
        elif tok.startswith("-") and tok not in known_flags:
            skip = True
        else:
            clean.append(tok)

    main(_args=parser.parse_args(clean))


if __name__ == "__main__":
    _cli_main()

rubin_sim.maf available – running full MAF suite.

══════════════════════════════════════════════════════════════════════
  MAF METRICS — ALL PARTIALLY-CLOUDY NIGHTS
  CSV: feb5_data.csv   DB: baseline_v5.1.0_10yrs.db
══════════════════════════════════════════════════════════════════════

Indexing DREAM CSV: feb5_data.csv
  220656 cloud_sys frames across 189 nights:
    2025-06-25  (432 frames)
    2025-06-26  (1536 frames)
    2025-06-27  (1248 frames)
    2025-06-28  (1559 frames)
    2025-06-29  (1583 frames)
    2025-06-30  (1396 frames)
    2025-07-01  (1377 frames)
    2025-07-02  (1376 frames)
    2025-07-03  (1375 frames)
    2025-07-04  (1375 frames)
    2025-07-05  (495 frames)
    2025-07-06  (1371 frames)
    2025-07-07  (1371 frames)
    2025-07-08  (942 frames)
    2025-07-09  (1152 frames)
    2025-07-10  (1369 frames)
    2025-07-11  (1367 frames)
    2025-07-12  (1375 frames)
    2025-07-13  (1060 frames)
    2025-07-14  (1304 frames)
    2025-07-15  (1361 frames)
    

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import sys

df = pd.read_csv('./maf_multinight_results/all_nights_summary.csv')

outdir = "analysis_plots"
os.makedirs(outdir, exist_ok=True)

strategies = df["strategy"].unique()

# ------------------------------------------------------------------
# Automatically determine numeric metrics to analyze
# (exclude identifiers / labels)
# ------------------------------------------------------------------
exclude_cols = ["night", "strategy", "strategy_label"]
metrics = [c for c in df.columns if c not in exclude_cols]

print("Metrics found:")
for m in metrics:
    print("  ", m)

for metric in metrics:
    if not np.issubdtype(df[metric].dtype, np.number):
        continue

    plt.figure(figsize=(6,4))

    for strat in strategies:
        vals = df[df.strategy == strat][metric].dropna()
        plt.hist(vals, bins=20, alpha=0.5, label=strat)

    plt.xlabel(metric)
    plt.ylabel("Number of Nights")
    plt.title(f"{metric} Distribution")
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"{outdir}/hist_{metric}.png", dpi=150)
    plt.close()

summary_metrics = [
    "Total_photons",
    "Median_m5",
    "Open_shutter_frac",
    "Median_airmass",
    "Mean_slew_s",
    "Sky_frac_visited",
]

fig, axes = plt.subplots(2,3, figsize=(12,7))
axes = axes.flatten()

for ax, metric in zip(axes, summary_metrics):
    medians = [
        df[df.strategy == strat][metric].median()
        for strat in strategies
    ]
    ax.bar(strategies, medians)
    ax.set_title(metric)
    ax.set_ylabel("Median Across Nights")

plt.tight_layout()
plt.savefig(f"{outdir}/six_panel_summary.png", dpi=150)
plt.close()

plt.figure(figsize=(7,5))

for strat in strategies:
    sub = df[df.strategy == strat].sort_values("night")
    cumulative = np.cumsum(sub["Total_photons"].values)
    plt.plot(sub["night"], cumulative, marker="o", label=strat)

plt.ylabel("Cumulative Photons")
plt.xlabel("Night")
plt.title("Cumulative Photon Collection")
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.savefig(f"{outdir}/cumulative_photons.png", dpi=150)
plt.close()

Metrics found:
   N_visits
   Total_photons
   Open_shutter_frac
   Median_m5
   Mean_m5
   Coadd_m5_proxy
   Median_airmass
   Mean_cloud_ext
   Min_cloud_ext
   Mean_slew_s
   Total_slew_h
   Total_exposure_h
   Sky_frac_visited
   fO_area_deg2
   Delta_depth_vs_phot
   Max_gap_min
   Median_gap_min


In [7]:
plt.figure(figsize=(8,5))

for strat in strategies:
    sub = df[df.strategy == strat]
    plt.plot(sub["night"], sub["Median_m5"], marker="o", label=strat)

plt.ylabel("Per-Night Median m5 Depth")
plt.xlabel("Night")
plt.title("Nightly Depth Evolution")
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.savefig(f"{outdir}/per_night_depth_timeseries.png", dpi=150)
plt.close()

In [8]:
plt.figure(figsize=(7,5))

for strat in strategies:
    vals = df[df.strategy == strat]["Median_m5"].dropna()
    plt.hist(vals, bins=20, alpha=0.5, label=strat)

plt.xlabel("Per-Night Median m5 Depth")
plt.ylabel("Number of Nights")
plt.title("Distribution of Nightly Depth")
plt.legend()
plt.tight_layout()
plt.savefig(f"{outdir}/hist_per_night_depth.png", dpi=150)
plt.close()

In [9]:
plt.figure(figsize=(8,5))

for strat in strategies:
    sub = df[df.strategy == strat]
    plt.plot(sub["night"], sub["Total_photons"], marker="o", label=strat)

plt.ylabel("Photons Collected Per Night")
plt.xlabel("Night")
plt.title("Nightly Photon Collection")
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.savefig(f"{outdir}/per_night_photons.png", dpi=150)
plt.close()

In [10]:
plt.figure(figsize=(7,5))

for strat in strategies:
    vals = df[df.strategy == strat]["Total_photons"].dropna()
    plt.hist(vals, bins=20, alpha=0.5, label=strat)

plt.xlabel("Photons Per Night")
plt.ylabel("Number of Nights")
plt.title("Distribution of Nightly Photon Yield")
plt.legend()
plt.tight_layout()
plt.savefig(f"{outdir}/hist_per_night_photons.png", dpi=150)
plt.close()

In [11]:
plt.figure(figsize=(7,5))

for strat in strategies:
    sub = df[df.strategy == strat]
    plt.scatter(sub["Median_m5"], sub["Total_photons"], alpha=0.6, label=strat)

plt.xlabel("Nightly Median m5")
plt.ylabel("Photons Per Night")
plt.title("Depth vs Photon Efficiency")
plt.legend()
plt.tight_layout()
plt.savefig(f"{outdir}/depth_vs_photons.png", dpi=150)
plt.close()

In [16]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CSV_PATH = "./maf_multinight_results/all_nights_summary.csv"   # your file
OUTDIR = "allmetrics"
os.makedirs(OUTDIR, exist_ok=True)

# Map "logical" metric names → actual CSV column names
METRIC_COLS = {
    "N_visits": "N_visits",
    "Total_photons": "Total_photons",
    "Open_shutter_frac": "Open_shutter_frac",
    "Median_m5": "Median_m5",
    "Mean_m5": "Mean_m5",
    "Coadd_m5_proxy": "Coadd_m5_proxy",
    "Median_airmass": "Median_airmass",
    "Mean_cloud_ext": "Mean_cloud_ext",
    "Min_cloud_ext": "Min_cloud_ext",
    "Mean_slew_s": "Mean_slew_s",
    "Total_slew_h": "Total_slew_h",
    "Total_exposure_h": "Total_exposure_h",
    "Sky_frac_visited": "Sky_frac_visited",
    "fO_area_deg2": "fO_area_deg2",
    "Delta_depth_vs_phot": "Delta_depth_vs_phot",
    "Max_gap_min": "Max_gap_min",
    "Median_gap_min": "Median_gap_min",
}

STRATEGIES = ["absolute", "motion", "scheduler"]
STRAT_COLORS = {
    "absolute": "#2ca02c",
    "motion": "#1f77b4",
    "scheduler": "#d62728",
}
STRAT_LABELS = {
    "absolute": "Abs Min (Green)",
    "motion": "Motion Track (Blue)",
    "scheduler": "Scheduler (Red)",
}

df = pd.read_csv(CSV_PATH)

# light sanity
for col in ["night", "strategy"]:
    if col not in df.columns:
        raise RuntimeError(f"Expected column '{col}' not found in CSV")

# also ensure metric columns exist
missing_cols = [c for c in METRIC_COLS.values() if c not in df.columns]
if missing_cols:
    raise RuntimeError(f"Metric columns missing from CSV: {missing_cols}")

for metric_label, colname in METRIC_COLS.items():
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    ax_hist, ax_bar = axes

    # collect finite values per strategy
    data_by_s = {}
    for s in STRATEGIES:
        vals = df.loc[df["strategy"] == s, colname].astype(float)
        vals = vals.replace([np.inf, -np.inf], np.nan).dropna()
        if not vals.empty:
            data_by_s[s] = vals.values

    if not data_by_s:
        plt.close(fig)
        continue

    all_vals = np.concatenate(list(data_by_s.values()))
    if np.nanmax(all_vals) == np.nanmin(all_vals):
        bins = 10
    else:
        bins = max(10, int(np.sqrt(all_vals.size)))

    # left: histogram over nights, all strategies
    for s, vals in data_by_s.items():
        ax_hist.hist(
            vals,
            bins=bins,
            alpha=0.5,
            color=STRAT_COLORS.get(s, "gray"),
            label=STRAT_LABELS.get(s, s),
            edgecolor="black",
            linewidth=0.7,
        )

    ax_hist.set_title(f"{metric_label} per night", fontsize=11)
    ax_hist.set_xlabel(metric_label)
    ax_hist.set_ylabel("Number of nights")
    ax_hist.legend(fontsize=8)
    ax_hist.grid(alpha=0.3)

    # right: mean ± std per strategy
    means, stds, labels, colors = [], [], [], []
    for s in STRATEGIES:
        vals = data_by_s.get(s)
        if vals is None or len(vals) == 0:
            continue
        means.append(np.mean(vals))
        stds.append(np.std(vals))
        labels.append(STRAT_LABELS.get(s, s))
        colors.append(STRAT_COLORS.get(s, "gray"))

    if means:
        x = np.arange(len(means))
        ax_bar.bar(
            x,
            means,
            yerr=stds,
            color=colors,
            alpha=0.8,
            edgecolor="black",
            linewidth=0.8,
            capsize=4,
        )
        ax_bar.set_xticks(x)
        ax_bar.set_xticklabels(labels, rotation=15, ha="right", fontsize=8)
        ax_bar.set_ylabel(metric_label)
        ax_bar.set_title(f"{metric_label} mean ± std across nights", fontsize=11)
        ax_bar.grid(axis="y", alpha=0.3)
    else:
        ax_bar.set_visible(False)

    plt.tight_layout()
    outpath = os.path.join(OUTDIR, f"{metric_label}_hist_and_means.png")
    plt.savefig(outpath, dpi=150, bbox_inches="tight")
    plt.close(fig)

print(f"Saved per-metric plots under {os.path.abspath(OUTDIR)}")


Saved per-metric plots under /home/s/segev/notebooks/contrails/allmetrics


In [21]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os

# Load data
df = pd.read_csv("./maf_multinight_results/all_nights_summary.csv")

# Create output directory
output_dir = "maf_plots"
os.makedirs(output_dir, exist_ok=True)

plt.style.use('default')
plt.rcParams.update({'font.size': 10, 'axes.linewidth': 0.8})

metrics = {
    'N_visits': {'label': 'Number of visits', 'units': '', 'zp': 0},
    'Total_photons': {'label': 'Total photons', 'units': '', 'zp': 0},
    'Open_shutter_frac': {'label': 'Open shutter fraction', 'units': '', 'zp': 0},
    'Median_m5': {'label': 'Median $m_5$', 'units': ' (mag)', 'zp': 27.4},
    'Mean_m5': {'label': 'Mean $m_5$', 'units': ' (mag)', 'zp': 27.4},
    'Coadd_m5_proxy': {'label': 'Coadd $m_5$', 'units': ' (mag)', 'zp': 27.4},
    'Median_airmass': {'label': 'Median airmass', 'units': '', 'zp': 0},
    'Mean_cloud_ext': {'label': 'Mean cloud extinction', 'units': ' (mag)', 'zp': 0},
    'Min_cloud_ext': {'label': 'Minimum cloud extinction', 'units': ' (mag)', 'zp': 0},
    'Mean_slew_s': {'label': 'Mean slew time', 'units': ' (s)', 'zp': 0},
    'Total_slew_h': {'label': 'Total slew time', 'units': ' (h)', 'zp': 0},
    'Total_exposure_h': {'label': 'Total exposure time', 'units': ' (h)', 'zp': 0},
    'Sky_frac_visited': {'label': 'Sky fraction visited', 'units': '', 'zp': 0},
    'fO_area_deg2': {'label': 'f_O area', 'units': ' (deg²)', 'zp': 0},
    'Delta_depth_vs_phot': {'label': 'Δdepth vs photons', 'units': ' (mag)', 'zp': 0},
    'Max_gap_min': {'label': 'Maximum gap', 'units': ' (min)', 'zp': 0},
    'Median_gap_min': {'label': 'Median gap', 'units': ' (min)', 'zp': 0}
}

# Generate and SAVE individual plot for each metric
for i, (metric, info) in enumerate(metrics.items(), 1):
    values = df[metric].dropna().values
    values = values[np.isfinite(values)]
    if len(values) == 0:
        print(f"Skipping {metric}: no valid data")
        continue
    
    values -= info['zp']
    
    fig, ax = plt.subplots(figsize=(6, 4.5))
    
    # Auto binning
    if np.ptp(values) > 0:
        bins = max(20, int(np.sqrt(len(values))))
    else:
        bins = 10
    
    # Main histogram
    ax.hist(values, bins=bins, color='0.7', edgecolor='black', linewidth=0.5)
    
    # Labels with units (LSST MAF paper style)
    xlabel = f"{info['label']}{info['units']}"
    ylabel = 'Number of nights'
    title = f"Multi-night metric: {metric}"
    
    ax.set_xlabel(xlabel, fontsize=12)
    ax.set_ylabel(ylabel, fontsize=12)
    ax.set_title(title)
    
    # MAF-style formatting
    ax.grid(alpha=0.3, linestyle=':')
    
    # Area-normalized overlay
    if len(values) > 0 and np.ptp(values) > 0:
        ax2_weights = np.full_like(values, 1000.0/len(values))
        ax.hist(values, bins=bins, weights=ax2_weights, 
                color='blue', alpha=0.3, histtype='step', linewidth=1.5)
        ax.text(0.02, 0.98, 'Area equiv. (1000s nights)', transform=ax.transAxes, 
                fontsize=10, color='blue', va='top')
    
    fig.tight_layout()
    
    # SAVE instead of show
    filename = f"{metric.replace('_', '-')}.png"
    filepath = os.path.join(output_dir, filename)
    plt.savefig(filepath, dpi=150, bbox_inches='tight')
    plt.close(fig)  # Close figure to free memory
    
    print(f"✅ Saved Plot {i}/{len(metrics)}: {filepath} ({len(values)} values)")

print(f"\n🎉 Saved ALL {i} LSST MAF-style plots to '{output_dir}' folder!")
print("Files saved as: N-visits.png, Coadd-m5-proxy.png, etc.")


✅ Saved Plot 1/17: maf_plots/N-visits.png (86 values)
✅ Saved Plot 2/17: maf_plots/Total-photons.png (86 values)
✅ Saved Plot 3/17: maf_plots/Open-shutter-frac.png (86 values)
✅ Saved Plot 4/17: maf_plots/Median-m5.png (86 values)
✅ Saved Plot 5/17: maf_plots/Mean-m5.png (86 values)
✅ Saved Plot 6/17: maf_plots/Coadd-m5-proxy.png (86 values)
✅ Saved Plot 7/17: maf_plots/Median-airmass.png (86 values)
✅ Saved Plot 8/17: maf_plots/Mean-cloud-ext.png (86 values)
✅ Saved Plot 9/17: maf_plots/Min-cloud-ext.png (86 values)
✅ Saved Plot 10/17: maf_plots/Mean-slew-s.png (86 values)
✅ Saved Plot 11/17: maf_plots/Total-slew-h.png (86 values)
✅ Saved Plot 12/17: maf_plots/Total-exposure-h.png (86 values)
✅ Saved Plot 13/17: maf_plots/Sky-frac-visited.png (86 values)
✅ Saved Plot 14/17: maf_plots/fO-area-deg2.png (86 values)
✅ Saved Plot 15/17: maf_plots/Delta-depth-vs-phot.png (86 values)
✅ Saved Plot 16/17: maf_plots/Max-gap-min.png (86 values)
✅ Saved Plot 17/17: maf_plots/Median-gap-min.png (8

In [24]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CSV_PATH = "./maf_multinight_results/all_nights_summary.csv"   # your file
OUTDIR = "allmetrics"
os.makedirs(OUTDIR, exist_ok=True)

# Metric definitions with HUMAN-READABLE labels + proper units
METRICS = {
    "N_visits": {"label": "Number of visits", "units": ""},
    "Total_photons": {"label": "Total photons", "units": ""},
    "Open_shutter_frac": {"label": "Open shutter fraction", "units": ""},
    "Median_m5": {"label": "Median m₅", "units": " (mag)"},
    "Mean_m5": {"label": "Mean m₅", "units": " (mag)"}, 
    "Coadd_m5_proxy": {"label": "Coadd m₅ proxy", "units": " (mag)"},
    "Median_airmass": {"label": "Median airmass", "units": ""},
    "Mean_cloud_ext": {"label": "Mean cloud extinction", "units": " (mag)"},
    "Min_cloud_ext": {"label": "Minimum cloud extinction", "units": " (mag)"},
    "Mean_slew_s": {"label": "Mean slew time", "units": " (s)"},
    "Total_slew_h": {"label": "Total slew time", "units": " (h)"},
    "Total_exposure_h": {"label": "Total exposure time", "units": " (h)"},
    "Sky_frac_visited": {"label": "Sky fraction visited", "units": ""},
    "fO_area_deg2": {"label": "f_O area", "units": " (deg²)"},
    "Delta_depth_vs_phot": {"label": "Δdepth vs photons", "units": " (mag)"},
    "Max_gap_min": {"label": "Maximum gap", "units": " (min)"},
    "Median_gap_min": {"label": "Median gap", "units": " (min)"},
}

STRATEGIES = ["absolute", "motion", "scheduler"]
STRAT_COLORS = {
    "absolute": "#2ca02c",    # green
    "motion": "#1f77b4",      # blue  
    "scheduler": "#d62728",   # red
}
STRAT_LABELS = {
    "absolute": "Abs Min",
    "motion": "Motion Track", 
    "scheduler": "Scheduler",
}

df = pd.read_csv(CSV_PATH)

# Sanity checks
for col in ["night", "strategy"]:
    if col not in df.columns:
        raise RuntimeError(f"Expected column '{col}' not found in CSV")

missing_cols = [c for c in METRICS.keys() if c not in df.columns]
if missing_cols:
    raise RuntimeError(f"Metric columns missing: {missing_cols}")

# LSST MAF publication style - nicer fonts, no bold
plt.rcParams.update({
    'font.family': 'serif',      # nicer serif font
    'font.size': 11, 
    'axes.linewidth': 0.8,
    'figure.dpi': 150,
    'mathtext.fontset': 'stix',  # better math rendering
})

for metric_key, info in METRICS.items():
    colname = metric_key  # direct mapping
    
    fig, ax = plt.subplots(figsize=(7, 5))  # SINGLE subplot only
    
    # Collect data per strategy
    data_by_s = {}
    for s in STRATEGIES:
        vals = df.loc[df["strategy"] == s, colname].astype(float)
        vals = vals.replace([np.inf, -np.inf], np.nan).dropna()
        if not vals.empty:
            data_by_s[s] = vals.values

    if not data_by_s:
        plt.close(fig)
        continue

    all_vals = np.concatenate(list(data_by_s.values()))
    
    # LSST MAF-style auto-binning
    if np.ptp(all_vals) > 0:
        bins = max(25, int(np.sqrt(all_vals.size)))
    else:
        bins = 15

    # Partially transparent overlapping histograms (3 strategies)
    for s, vals in data_by_s.items():
        ax.hist(
            vals,
            bins=bins,
            alpha=0.65,  # partially transparent
            color=STRAT_COLORS[s],
            label=STRAT_LABELS[s],
            edgecolor="black",
            linewidth=0.8,
        )
    
    # HUMAN-READABLE axis labels + UNITS on BOTH axes - no bold, nicer font
    xlabel = f"{info['label']}{info['units']}"
    ylabel = "Number of nights"
    
    ax.set_xlabel(xlabel, fontsize=12)
    ax.set_ylabel(ylabel, fontsize=12)
    ax.set_title(f"Multi-night {info['label']}", fontsize=13)
    
    # LSST MAF paper styling
    ax.legend(loc='best', fontsize=10)
    ax.grid(alpha=0.3, linestyle=':', linewidth=0.8)
    
    # Tight layout for publication
    plt.tight_layout()
    
    # Save
    outpath = os.path.join(OUTDIR, f"{metric_key}_comparison.png")
    plt.savefig(outpath, dpi=150, bbox_inches="tight")
    plt.close(fig)
    
    print(f"✅ Saved: {metric_key}_comparison.png ({len(all_vals)} nights)")

print(f"\n🎉 Saved 17 LSST MAF-style plots to '{os.path.abspath(OUTDIR)}'")
print("✓ Single subplot per metric")
print("✓ Human-readable labels with units") 
print("✓ Serif fonts, no bold, publication quality")
print("✓ 3-color partially transparent histograms")


✅ Saved: N_visits_comparison.png (86 nights)
✅ Saved: Total_photons_comparison.png (86 nights)
✅ Saved: Open_shutter_frac_comparison.png (86 nights)
✅ Saved: Median_m5_comparison.png (86 nights)
✅ Saved: Mean_m5_comparison.png (86 nights)
✅ Saved: Coadd_m5_proxy_comparison.png (86 nights)
✅ Saved: Median_airmass_comparison.png (86 nights)
✅ Saved: Mean_cloud_ext_comparison.png (86 nights)
✅ Saved: Min_cloud_ext_comparison.png (86 nights)
✅ Saved: Mean_slew_s_comparison.png (86 nights)
✅ Saved: Total_slew_h_comparison.png (86 nights)
✅ Saved: Total_exposure_h_comparison.png (86 nights)
✅ Saved: Sky_frac_visited_comparison.png (86 nights)
✅ Saved: fO_area_deg2_comparison.png (86 nights)
✅ Saved: Delta_depth_vs_phot_comparison.png (86 nights)
✅ Saved: Max_gap_min_comparison.png (86 nights)
✅ Saved: Median_gap_min_comparison.png (86 nights)

🎉 Saved 17 LSST MAF-style plots to '/home/s/segev/notebooks/contrails/allmetrics'
✓ Single subplot per metric
✓ Human-readable labels with units
✓ Ser